**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/shukla-2026"], check=True)
    os.chdir("/content/encore/entries/shukla-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Shukla, Rivera, Bladon & Jadhav (2026) from dandiset 001845

Paper: Shukla A, Rivera EL, Bladon JH, Jadhav SP (2026). A Novel Cooperative Behavior Assay in Rats Reveals
Distinct Social Coordination Strategies of Increasing Complexity and Social Reciprocity Deficits in a Model
of Fragile X Syndrome. J Neurosci 46(24) e0143262026, doi:10.1523/jneurosci.0143-26.2026.

Dataset: DANDI:001845, "Cooperative Behavior in WT and Fmr1 Rat Dyads", version 0.260526.1807
(doi:10.48324/dandi.001845/0.260526.1807), streamed from the archive.

Version. The paper cites the dandiset without a version. The paper's code repository
(JadhavLab/Social_CoOp_Behavior, README only) names version 0.260519.1308, an early partial upload with
1,007 of the 2,649 assets. The most recent published version, 0.260526.1807, is pinned: it holds every
subject and session the claims file describes that are in the dandiset at all, and the data check below
shows that every asset of 0.260519.1308 is in it unchanged (same blob).

Claims: `claims/001845/10.1523-jneurosci.0143-26.2026.yaml`, written from the version of record (Europe PMC
full text, main text, Methods and legends; the supplement was not available to the readers).

| Claim | Tested here |
|---|---|
| fig1e-match-rate-wt-higher | tested |
| fig1f-transition-rate-wt-higher | tested |
| fig1g-match-transition-regression | tested |
| fig1h-above-chance-late | tested |
| fig1i-performance-wt-higher | tested |
| fig1j-partner-location-predicts-choice | tested |
| figs1-mixed-pairs-impaired | blocked, data missing (no mixed pairs) |
| fig2b-opaque-barrier-lowers-performance | blocked, metadata insufficient (no session is labelled sensory deprived; the unanalysed sessions are surveyed) |
| fig2c-opaque-barrier-mixed-pairs | blocked, data missing |
| fig2e-visual-cue-learning-similar | blocked, data missing (no visual-cue task) |
| fig2f-cue-latency-wt-shorter | blocked, data missing |
| fig3c-triplets-by-contingency | tested |
| fig3d-entropy-wt-lower | tested |
| fig3e-exploration-efficiency-wt-higher | tested |
| fig3g-virtual-agent-performance-similar | blocked, data missing (no virtual-agent task) |
| fig3h-virtual-agent-efficiency-similar | blocked, data missing |
| figs2-jaccard-similarity | tested (100% and 50%; no sensory-deprived sessions) |
| fig4b-arrival-lag-wt-shorter | tested |
| fig4c-departure-lag-wt-shorter | tested |
| fig4d-dwell-time | tested |
| fig4eg-cross-correlation-wt-higher | tested |
| fig4h-peak-lag-distribution | tested |
| fig4i-peak-correlation-wt-higher | tested |
| fig5d-leading-near-half | tested |
| fig5e-leading-difference-50 | tested |
| figs3-leadership-history | blocked by the claims file, methods underspecified (readings shown) |
| fig6-marl-partner-aware | blocked, methods underspecified (simulation, no code) |
| fig7b-fmr1-more-asynchronous-matches | tested |
| fig7c-trial-duration-vs-performance | tested |
| fig7e-gaze-final-quartile | tested |
| fig7g-gaze-100-late | tested |
| fig7h-gaze-50-fmr1-follower | tested |
| fig8b-partner-history-cooperation | tested; blocked, the test is decided by how follower trials are built |
| fig8d-fmr1-longer-delta-t | tested; blocked, depends on fig8b |
| fig8e-glmm-partner-cooperation | tested; blocked, depends on fig8b |

The streaming load takes about four minutes; the whole notebook 10 to 13 minutes (the run time is printed at the end).

Revised after the first verification round (transcripts/001845/verifier-1-report.md): pixel scale and
position checks, follower trials as the Methods describe them with a shifted null, the arrival-lag
outlier, the rebuilt matches of Figure 1H, and a survey of the sessions not analysed.

Revised after the second round (transcripts/001845/verifier-2-report.md): the fits of Figure 1G with the raw
sequences, per-session median lags and reading (a) in the aligned sessions for Figure 4B and 4C, checks in
the aligned sessions for Figures 1J, 7E, 7G and 7H, and corrected wording. Two changes of outcome were
proposed (fig4b-arrival-lag-wt-shorter and fig4c-departure-lag-wt-shorter, from blocked to not
reproduced).

Revised after the third round (transcripts/001845/verifier-3-report.md): the verifier agreed with both
proposed changes, and fig4b and fig4c are now not reproduced.

In [ ]:
import numpy as np, pandas as pd, collections, time, warnings
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
import coop
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
T_START = time.time()

In [ ]:
assets = coop.list_assets()
older = coop.list_assets("0.260519.1308")
blob = {a["path"]: a["blob"] for a in assets}
same = sum(blob.get(a["path"]) == a["blob"] for a in older)
print(f"version {coop.VERSION}: {len(assets)} assets ({sum(a['path'].endswith('.nwb') for a in assets)} NWB, {sum(a['path'].endswith('.mp4') for a in assets)} videos)")
print(f"version 0.260519.1308: {len(older)} assets, of which {same} are in {coop.VERSION} with the same blob")
tab = pd.DataFrame(coop.session_table(assets))
print("NWB files per subject and contingency in the pinned version:")
print(tab.groupby(["subject", "contingency"]).size().unstack().to_string())
missing = {}
for (subj, c), d in tab.groupby(["subject", "contingency"]):
    m = sorted(set(range(1, d.session.max() + 1)) - set(d.session))
    if m:
        missing[(subj, c)] = m
print("session numbers missing from a subject's sequence:", missing)

In [ ]:
rows, files, S = coop.load()
import re
print(collections.Counter(re.sub(r"subject \S+ paired with \S+", "subject X paired with Y", f["description"]) for f in files))
print(f"streamed {len(files)} NWB files of dandiset {coop.DANDISET} version {coop.VERSION}")

In [ ]:
# Counts by subject, pair, genotype, contingency
subj = pd.DataFrame([dict(subject=f["subject"], genotype=f["genotype"], side=f["side"]) for f in files])
print("files per subject and genotype:")
print(subj.groupby(["subject", "genotype"]).size().to_string())
print()
pairs = pd.DataFrame([dict(pair="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"]) for s in S])
print("pair sessions analysed (first 21 at 100%, first 35 at 50%):")
print(pairs.groupby(["genotype", "pair", "contingency"]).size().unstack().to_string())
print()
print("both files of every pair session hold identical events:", all(s["events_identical"] for s in S))
print("every pair is of one genotype:", sorted(set(s["genotype"] for s in S)))
dur = np.array([(s["t1"] - s["t0"]) / 60 for s in S])
print(f"session (epoch) length: {dur.min():.1f} to {dur.max():.1f} min, median {np.median(dur):.1f} (paper: 20 to 30 min)")
print("camera metres per pixel in every file:", sorted(set(f["camera_m_per_px"] for f in files)))
npk = np.array([len(r["poke_t"]) for s in S for r in s["rats"].values()])
print(f"pokes per rat and session: {npk.min()} to {npk.max()}, median {np.median(npk):.0f}")
nm = np.array([len(s["match_t"]) for s in S])
print(f"matched pokes per pair session: {nm.min()} to {nm.max()}, median {np.median(nm):.0f}")
allpos = np.concatenate([r["pos"]["head"] for s in S for r in s["rats"].values()])
print(f"head position (cm): x {np.nanmin(allpos[:,0]):.1f} to {np.nanmax(allpos[:,0]):.1f}, y {np.nanmin(allpos[:,1]):.1f} to {np.nanmax(allpos[:,1]):.1f}; NaN fraction {np.isnan(allpos[:,0]).mean():.3f}")
fr = np.array([1 / np.median(np.diff(r["pos_t"])) for s in S for r in s["rats"].values()])
print(f"position frame rate: {fr.min():.2f} to {fr.max():.2f} Hz (paper: 30 fps)")
inside = np.mean([(r["poke_t"].min() >= s["t0"]) & (r["poke_t"].max() <= s["t1"]) for s in S for r in s["rats"].values() if len(r["poke_t"])])
print(f"fraction of rat sessions whose pokes all fall inside the epoch: {inside:.3f}")
ndrop = sum(r["n_dropped"] for s in S for r in s["rats"].values()); nall = sum(len(r["raw_poke_t"]) for s in S for r in s["rats"].values())
print(f"pokes within 0.5 s of a poke at another well (dropped as crosstalk): {ndrop} of {nall} ({ndrop / nall:.3f})")
ch = pd.DataFrame([dict(pair="-".join(s["pair"]), c=s["contingency"], rank=s["rank"], side=side, raw=len(r["raw_vw"]) - 1, clean=len(r["vw"]) - 1, dropped=r["n_dropped"]) for s in S for side, r in s["rats"].items()])
print(f"rat sessions changed by the cleaning: {(ch.raw != ch.clean).sum()} of {len(ch)}; transitions per rat session raw max {ch.raw.max()}, cleaned max {ch.clean.max()}")
print(ch.sort_values("raw", ascending=False).head(8).to_string())
consec = sum(int(np.sum(s["match_w"][1:] == s["match_w"][:-1])) for s in S)
print("consecutive matched pokes at the same well:", consec, "of", sum(max(len(s['match_w']) - 1, 0) for s in S))
rw = pd.DataFrame([dict(c=s["contingency"], m=len(s["match_t"]), r=len(s["rewarded_t"])) for s in S]).groupby("c").sum()
rw["rewarded fraction"] = rw.r / rw.m
print(rw.to_string())

### Data checks

Position and pokes share one clock. If they do, the head is at a well at the moment of a poke at that
well and is not there at the same moment shifted by some seconds. The well centre is the median head
position at the rat's pokes on that well in that session.

In [ ]:
def dist_at(s, side, dt):
    r = s["rats"][side]
    c = coop.well_centres(r["pos"]["head"], r["pos_t"], r["poke_t"], r["poke_w"])
    idx = np.clip(np.searchsorted(r["pos_t"], r["poke_t"] + dt), 0, len(r["pos_t"]) - 1)
    p = r["pos"]["head"][idx]
    cc = np.array([c[w] for w in r["poke_w"]])
    return np.linalg.norm(p - cc, axis=1)

for dt in (0, 2, 5, 10, -5):
    d = np.concatenate([dist_at(s, side, dt) for s in S[::7] for side in ("left", "right")])
    print(f"poke times shifted by {dt:+d} s: median head distance from the poked well {np.nanmedian(d):.1f} cm, within 6.4 cm {np.nanmean(d < coop.RADIUS_CM):.2f}")
# pixel scale: processed head (cm) regressed on the raw DeepLabCut head (pixels) in each file
scale = pd.DataFrame([dict(subject=f["subject"], cm_per_px=f["cm_per_px"]) for f in files])
print("cm per raw pixel by subject (median, min, max over files; files without raw pose):")
print(scale.groupby("subject").cm_per_px.agg(["median", "min", "max", lambda x: int(x.isna().sum())]).round(4).to_string())
print("camera device in every file:", sorted(set(f["camera_m_per_px"] for f in files)), "m per pixel")
fr_in = []
for s in S:
    for side, r in s["rats"].items():
        d = dist_at(s, side, 0)
        ok = np.isfinite(d)
        fr_in.append(dict(subject=r["subject"], cm_per_px=r["cm_per_px"], no_position=1 - ok.mean(),
                          **{f"{px} px": np.mean(d[ok] < px * r["cm_per_px"]) if ok.any() else np.nan for px in (20, 40, 60)},
                          **{f"{cm} cm": np.mean(d[ok] < cm) if ok.any() else np.nan for cm in (6.4, 10, 20)}))
fr_in = pd.DataFrame(fr_in)
print("head position at the poke times: fraction with no position (NaN), and fraction of the others within a radius")
print("of the poked well's centre, mean over rat sessions; radius in each video's own pixels (subject scale) or in cm:")
print(fr_in.groupby("subject").mean().round(3).to_string())
print(fr_in.drop(columns="subject").mean().round(3).to_string())
# per rat session: fraction of poke-time head positions within 20 cm of the poked well, at the poke time and
# with the poke times shifted by -30 to +30 s (a clock offset would show as a better fit at some shift)
AL = []
for s in S:
    for side, r in s["rats"].items():
        f = {dt: np.nanmean(dist_at(s, side, dt) < 20) for dt in range(-30, 31, 5)}
        best = max(f, key=f.get)
        AL.append(dict(pair="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], side=side, within20=f[0], best_shift=best, within20_best=f[best]))
AL = pd.DataFrame(AL)
print("rat sessions by fraction of poke-time head positions within 20 cm of the poked well:", np.histogram(AL.within20, bins=[0, 0.2, 0.4, 0.6, 0.8, 0.9, 1.0001])[0].tolist(), "(bins 0, 0.2, 0.4, 0.6, 0.8, 0.9, 1)")
print(AL.groupby(["genotype", "pair"]).agg(rat_sessions=("within20", "size"), within20_median=("within20", "median"), within20_min=("within20", "min"),
                                           at_least_090=("within20", lambda x: int((x >= 0.9).sum())), best_shift_median=("best_shift", "median"),
                                           within20_at_best_shift=("within20_best", "median")).round(3).to_string())
# a pair session is aligned when both rats have at least 90% of poke-time head positions within 20 cm of the poked well
ALIGNED = set(k for k, d in AL.groupby(["pair", "contingency", "rank"]) if (d.within20 >= 0.9).all())
print(f"aligned pair sessions: {len(ALIGNED)} of {len(S)}")
cent = []
for s in S:
    for side, r in s["rats"].items():
        c = coop.well_centres(r["pos"]["head"], r["pos_t"], r["poke_t"], r["poke_w"])
        cent.append(dict(side=side, **{f"w{k}x": c[k][0] for k in c}, **{f"w{k}y": c[k][1] for k in c}))
cent = pd.DataFrame(cent)
print("well centres (median head position at pokes, cm), median over sessions by maze:")
print(cent.groupby("side").median().round(1).to_string())
d12 = np.hypot(cent.w1x - cent.w2x, cent.w1y - cent.w2y)
print(f"distance between wells 1 and 2: median {np.nanmedian(d12):.1f} cm (maze 80 x 80 cm)")
tv = np.concatenate([np.asarray(r["table"]["stop_time"]) - np.asarray(r["table"]["start_time"]) for s in S for r in s["rats"].values()])
print(f"dataset reward_well_visits table (position based, 20 cm): {len(tv)} visits, duration median {np.median(tv):.1f} s")

### Per pair and per rat measures

A visit is a run of consecutive pokes at one well; a transition is a visit to a well other than the
previous one (claims file, opening comment). Matches are the dataset's matched pokes.
Performance is 200 x matches / transitions of both rats (Eq. 1 as the claims file reads it).

In [ ]:
P, R = [], []
for s in S:
    mins = (s["t1"] - s["t0"]) / 60
    tr = {side: max(len(r["vw"]) - 1, 0) for side, r in s["rats"].items()}
    trr = {side: max(len(r["raw_vw"]) - 1, 0) for side, r in s["rats"].items()}
    m = len(s["match_t"])
    P.append(dict(unit="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"],
                  matches=m, minutes=mins, match_r1=m / mins, match_r2=m, transitions=tr["left"] + tr["right"],
                  trans_r1=(tr["left"] + tr["right"]) / mins, trans_r2=tr["left"] + tr["right"],
                  trans_raw_r1=(trr["left"] + trr["right"]) / mins, trans_raw_r2=trr["left"] + trr["right"],
                  performance=200 * m / max(tr["left"] + tr["right"], 1),
                  performance_raw=200 * m / max(trr["left"] + trr["right"], 1)))
    for side, r in s["rats"].items():
        R.append(dict(unit=r["subject"], pair="-".join(s["pair"]), genotype=r["genotype"], contingency=s["contingency"],
                      rank=s["rank"], side=side, transitions=tr[side], trans_r1=tr[side] / mins, trans_r2=tr[side],
                      trans_raw_r1=trr[side] / mins, trans_raw_r2=trr[side],
                      match_r1=m / mins, match_r2=m, minutes=mins))
P = pd.DataFrame(P)
R = pd.DataFrame(R)
print(P.groupby(["contingency", "genotype"])[["matches", "transitions", "performance", "minutes"]].describe().T.round(2).to_string())

def genotype_tests(df, dv, cont):
    d = df[df.contingency == cont]
    a = coop.anova_A(d, dv)
    b = coop.anova_B(d, dv)
    means = d.groupby("genotype")[dv].mean()
    return dict(dv=dv, contingency=cont, WT=means.get("WT"), Fmr1=means.get("Fmr1"),
                A_F=a["genotype"][0], A_df=f"{a['genotype'][1]},{a['genotype'][2]}", A_p=a["genotype"][3],
                A_int_p=a["interaction"][3],
                B_F=b["genotype"][0], B_df=f"{b['genotype'][1]},{b['genotype'][2]}", B_p=b["genotype"][3],
                B_int_p=b["interaction"][3])

def per_unit_means(df, dv, cont):
    return df[df.contingency == cont].groupby(["genotype", "unit"])[dv].mean().round(3).to_dict()

## fig1e-match-rate-wt-higher

> WT pairs reach significantly higher match (coordination) rates than Fmr1 pairs under both the 100% and
> the 50% reward contingency.

Rate readings r1 (matches per minute of session) and r2 (matches per session); ANOVA readings A
(ordinary two-way, as the paper's df show) and B (mixed design).

In [ ]:
t1e = pd.DataFrame([genotype_tests(P, dv, c) for dv in ("match_r1", "match_r2") for c in ("rc100", "rc50")])
print(t1e.round(4).to_string())
for c in ("rc100", "rc50"):
    print(c, "per pair mean matches per minute:", per_unit_means(P, "match_r1", c))

**Result: blocked (methods underspecified).** WT pairs make more matches than Fmr1 pairs in both contingencies (per minute 1.31 against 0.86 at 100%, 2.09 against 1.53 at 50%). Under reading A, the ordinary two-way ANOVA whose df match the paper's (F(1,168) and F(1,280)), the genotype effect is significant in both contingencies and under both rate readings (p 0.0004 and below 0.0001 per minute; 0.0015 and 0.0006 per session). Under reading B, the mixed-design ANOVA with pair as subject, it is not (p 0.18 to 0.45, F(1,8)). The claims file blocks a claim where the two readings disagree.

## fig1f-transition-rate-wt-higher

> Fmr1 rats make fewer task-related arm transitions than WT rats (they are slower in the task), under
> both contingencies.

In [ ]:
t1f = pd.DataFrame([genotype_tests(R, dv, c) for dv in ("trans_r1", "trans_r2", "trans_raw_r1", "trans_raw_r2") for c in ("rc100", "rc50")])
print(t1f.round(5).to_string())
for c in ("rc100", "rc50"):
    print(c, "per rat mean transitions per minute:", per_unit_means(R, "trans_r1", c))

**Result: blocked (methods underspecified).** WT rats make more transitions than Fmr1 rats (per minute 1.98 against 1.49 at 100%, 3.02 against 2.47 at 50%). Reading A gives a significant genotype effect in both contingencies under both rate readings (p from below 0.0001 to 0.00055, df 1,378 and 1,630, the paper's), and so do the raw, uncleaned sequences (p from below 0.00001 to 0.027). Reading B, with rat as subject, does not (p 0.07 to 0.42). The readings disagree.

## fig1g-match-transition-regression

> In the 50% condition, match rate rises linearly with transition rate in both genotypes, with a steeper
> slope and a better fit in WT (beta 0.79, R2 0.74) than in Fmr1 (beta 0.64, R2 0.44).

Readings of the points: (a) pair and session, the pair's summed transitions; (b) rat and session, the
rat's transitions against its pair's matches; (c) session, pairs averaged; each under r1 and r2.

In [ ]:
rows1g = []
for reading, df in (("a", P), ("b", R)):
    for r in ("r1", "r2"):
        for g in ("WT", "Fmr1"):
            d = df[(df.contingency == "rc50") & (df.genotype == g)]
            lr = stats.linregress(d[f"trans_{r}"], d[f"match_{r}"])
            rows1g.append(dict(points=reading, rate=r, genotype=g, n=len(d), slope=lr.slope, R2=lr.rvalue ** 2, p=lr.pvalue))
for r in ("r1", "r2"):
    for g in ("WT", "Fmr1"):
        d = P[(P.contingency == "rc50") & (P.genotype == g)].groupby("rank")[[f"trans_{r}", f"match_{r}"]].mean()
        lr = stats.linregress(d[f"trans_{r}"], d[f"match_{r}"])
        rows1g.append(dict(points="c", rate=r, genotype=g, n=len(d), slope=lr.slope, R2=lr.rvalue ** 2, p=lr.pvalue))
for reading, df in (("a", P), ("b", R)):
    for r in ("r1", "r2"):
        for g in ("WT", "Fmr1"):
            d = df[(df.contingency == "rc50") & (df.genotype == g)]
            lr = stats.linregress(d[f"trans_raw_{r}"], d[f"match_{r}"])
            rows1g.append(dict(points=reading + " raw", rate=r, genotype=g, n=len(d), slope=lr.slope, R2=lr.rvalue ** 2, p=lr.pvalue))
t1g = pd.DataFrame(rows1g)
t1g["slope_in_tol"] = [abs(s - (0.79 if g == "WT" else 0.64)) <= (0.079 if g == "WT" else 0.064) for s, g in zip(t1g.slope, t1g.genotype)]
t1g["R2_in_tol"] = [abs(s - (0.74 if g == "WT" else 0.44)) <= (0.074 if g == "WT" else 0.044) for s, g in zip(t1g.R2, t1g.genotype)]
print(t1g.round(4).to_string())
print("rows 'a raw' and 'b raw' use the transitions of the raw sequences (no crosstalk removal)")
print("ranges plotted: transitions per minute", P.trans_r1.min().round(2), "to", P.trans_r1.max().round(2), "; matches per minute", P.match_r1.min().round(2), "to", P.match_r1.max().round(2))
fig, ax = plt.subplots(figsize=(4, 3))
for g, col in (("WT", "k"), ("Fmr1", "r")):
    d = P[(P.contingency == "rc50") & (P.genotype == g)]
    ax.scatter(d.trans_r1, d.match_r1, s=6, c=col, label=g)
ax.set_xlabel("transitions / min (pair)"); ax.set_ylabel("matches / min"); ax.legend(); plt.show()

**Result: partially reproduced.** Under reading (b), one point per rat and session, both regressions are significant and WT has the steeper slope and better fit: per minute, WT slope 0.818 and R2 0.795, Fmr1 slope 0.649 and R2 0.684. Both slopes and the WT R2 are within tolerance; the Fmr1 R2 is not (0.684 against 0.44 ± 0.044). Under readings (a) and (c) the slopes are 0.39 to 0.47, out of tolerance. The result uses the cleaned choice sequences (an assumption, see the data check). With the raw sequences (rows 'a raw' and 'b raw') the fits are much weaker (slopes 0.027 to 0.068, R2 0.028 to 0.121), so the outcome rests on the crosstalk removal.

## fig1i-performance-wt-higher

> Cooperative performance (matches relative to transitions, Eq. 1) is significantly lower in Fmr1 pairs
> than in WT pairs under both contingencies, so their impairment is not only due to slower transitions.

In [ ]:
t1i = pd.DataFrame([genotype_tests(P, dv, c) for dv in ("performance", "performance_raw") for c in ("rc100", "rc50")])
print(t1i.round(5).to_string())
for c in ("rc100", "rc50"):
    print(c, "per pair mean performance (%):", per_unit_means(P, "performance", c))
print(f"performance range {P.performance.min():.1f} to {P.performance.max():.1f} %")

**Result: blocked (methods underspecified).** Performance is higher in WT pairs (53.9 against 46.4% at 100%, 64.2 against 58.5% at 50%). Reading A finds the genotype effect in both contingencies (p = 0.015 and 0.001); reading B does not (p = 0.40 and 0.43). With the raw sequences, reading A finds it at 100% (p = 0.016) but not at 50% (p = 0.097). The outcome rests on the crosstalk removal: with the raw sequences reading B fails in both contingencies too (p = 0.41 and 0.69), so both ANOVA readings fail at 50% and agree that the claim fails, which would give not reproduced.

## fig1h-above-chance-late

> Both WT and Fmr1 pairs coordinate no better than chance early in training, and above chance late in
> training, by a permutation test that circularly shifts one rat's sequence of well visits.

The dataset's matches cannot be recomputed under a time shift, so matches are rebuilt from the pokes
(`coop.reconstructed_matches`: overlapping visits at corresponding wells, a hold tolerance chosen as the one
whose counts fit the dataset's best, no match at the well of the previous match). The agreement of the rebuilt counts with the dataset's is
printed first. Observed and shifted performance both use the rebuilt rule; a variant takes the observed performance
from the dataset's matches (p_dataset_obs). Shifts are uniform over
the session length (claims file, the standard choice); 1,000 per session; p is the fraction of
shifts with performance at or above the observed one (Methods, Permutation testing).

In [ ]:
def vis(r):
    return (r["vw"], r["vs"], r["ve"])

given = np.array([len(s["match_t"]) for s in S])
fits = {}
for hold in (0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0):
    rb = np.array([coop.reconstructed_matches(vis(s["rats"]["left"]), vis(s["rats"]["right"]), hold=hold) for s in S])
    fits[hold] = np.corrcoef(rb, given)[0, 1]
    print(f"  hold {hold} s: r = {fits[hold]:.4f}, total ratio {rb.sum() / given.sum():.3f}")
HOLD = max(fits, key=fits.get)
print("hold used (largest r):", HOLD)
rebuilt = np.array([coop.reconstructed_matches(vis(s["rats"]["left"]), vis(s["rats"]["right"]), hold=HOLD) for s in S])
print(f"rebuilt against dataset matches over {len(S)} pair sessions: totals {rebuilt.sum()} and {given.sum()}, r = {np.corrcoef(rebuilt, given)[0, 1]:.3f}, median abs difference {np.median(np.abs(rebuilt - given)):.0f}")

def perm_test(s, shifted_rat, rng, n=1000):
    rats = {r["subject"]: r for r in s["rats"].values()}
    a, b = s["pair"]
    r1, r2 = (rats[b], rats[a]) if shifted_rat == "first" else (rats[a], rats[b])  # r2 is shifted
    trans = sum(max(len(r["vw"]) - 1, 0) for r in s["rats"].values())
    period = s["t1"] - s["t0"]
    nrb = coop.reconstructed_matches(vis(r1), vis(r2), hold=HOLD)
    obs = 200 * nrb / max(trans, 1)
    obs_ds = 200 * len(s["match_t"]) / max(trans, 1)
    null = np.array([200 * coop.reconstructed_matches(vis(r1), vis(r2), hold=HOLD, shift=sh, t0=s["t0"], period=period) / max(trans, 1)
                     for sh in rng.uniform(0, period, n)])
    return obs, null.mean(), np.mean(null >= obs), nrb, len(s["match_t"]), np.mean(null >= obs_ds)

rng = np.random.default_rng(1845)
idx = {(s["pair"], s["contingency"], s["rank"]): s for s in S}
H = []
for pair in sorted(set(s["pair"] for s in S)):
    for cont, rank, label in (("rc100", 1, "100% first"), ("rc100", 21, "100% last"), ("rc50", 1, "50% first"), ("rc50", 35, "50% last")):
        s = idx[(pair, cont, rank)]
        for which in ("first", "second"):
            obs, nullm, p, nrb, nds, p_ds = perm_test(s, which, rng)
            H.append(dict(pair="-".join(pair), genotype=s["genotype"], session=label, shifted=which, observed=obs, null_mean=nullm, p=p,
                          rebuilt_matches=nrb, dataset_matches=nds, p_dataset_obs=p_ds,
                          dataset_perf=P[(P.unit == "-".join(pair)) & (P.contingency == cont) & (P["rank"] == rank)].performance.iloc[0]))
H = pd.DataFrame(H)
print(H.round(3).to_string())

def judge_h(early, late, pcol="p"):
    out = {}
    for which in ("first", "second"):
        d = H[H.shifted == which]
        e = d[d.session.isin(early)].groupby(["genotype", "pair"])[pcol].apply(lambda x: bool((x >= 0.05).all()))
        l = d[d.session.isin(late)].groupby(["genotype", "pair"])[pcol].apply(lambda x: bool((x < 0.05).all()))
        both = (e & l).groupby("genotype").sum()
        out[which] = dict(early_ns=e.groupby("genotype").sum().to_dict(), late_sig=l.groupby("genotype").sum().to_dict(), both=both.to_dict())
    return out

print("reading (a) early = 100% first, late = 50% last:", judge_h(["100% first"], ["50% last"]))
print("reading (b) early = first of each contingency, late = last of each:", judge_h(["100% first", "50% first"], ["100% last", "50% last"]))
print("variant: observed performance from the dataset's matches, null from the rebuilt rule (p_dataset_obs)")
print("reading (a):", judge_h(["100% first"], ["50% last"], "p_dataset_obs"))
print("reading (b):", judge_h(["100% first", "50% first"], ["100% last", "50% last"], "p_dataset_obs"))

**Result: partially reproduced.** Under reading (a) (first 100% session early, last analysed 50% session late), with the first rat shifted, 4 of 5 WT and 3 of 5 Fmr1 pairs are at chance early and above chance late; with the second rat shifted, 5 of 5 WT and 3 of 5 Fmr1 pairs. The two Fmr1 pairs that fail are FXM108-FXM109 and XFN1-XFN3, not above chance in the last 50% session (p 0.17 to 0.18 and 0.061 to 0.086). Under reading (b), the first and last session of each contingency, at most 1 pair per genotype meets both conditions. The test uses matches rebuilt from the pokes with a 6 s hold, whose counts agree with the dataset's overall (r = 0.994), but in first 100% sessions the rebuilt rule finds more matches than the dataset holds (for example 12 against 3 for FXM101-FXM110, 7 against 1 for FXM201-FXM202). With the observed performance taken from the dataset's matches, reading (a) gives 5 of 5 WT and 4 of 5 Fmr1 pairs under both shifted-rat readings (FXM108-FXM109 is now above chance late, p = 0.012 and 0.017; XFN1-XFN3 is not, p = 0.061 and 0.086). This variant does not test the early part: in 5 of the 10 first 100% sessions the dataset holds 1 to 3 matches where the rebuilt rule, which also gives the null, finds 3 to 12, so p_dataset_obs is 1.000 and the pair is at chance early by construction. The outcome is partial under either version.

## fig1j-partner-location-predicts-choice

> A multinomial logistic regression predicts a rat's next choice from its partner's current location
> above chance in both genotypes, and more accurately in WT than in Fmr1 rats, under both contingencies.

Per rat and contingency, one sample per transition: the next well against the partner's current well
(the well of its most recent poke). Readings of the moment: (a) when the focal rat's head leaves its
well (first frame after its last poke more than 6.4 cm from the well centre); (b) its last poke
there; (c) its first poke at the next well. Pooling: p1, sessions of the contingency pooled; p2, per
session, averaged. Real accuracy: scikit-learn `LogisticRegression()` with fivefold
`StratifiedKFold` (Methods). The 1,000-shuffle null uses the class the fitted model predicts for each
partner well, which for a single one-hot predictor is the most frequent next well in the training
folds; the agreement of this shortcut with scikit-learn on the real folds is printed. Its folds
assign each class's samples to folds in turn.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

def samples(s, side, moment):
    r = s["rats"][side]
    q = s["rats"]["right" if side == "left" else "left"]
    if len(r["vw"]) < 2 or len(q["poke_t"]) == 0:
        return np.zeros(0, int), np.zeros(0, int)
    if moment == "a":
        c = coop.well_centres(r["pos"]["head"], r["pos_t"], r["poke_t"], r["poke_w"])
        tt = []
        for k in range(len(r["vw"]) - 1):
            i0 = np.searchsorted(r["pos_t"], r["ve"][k]); i1 = np.searchsorted(r["pos_t"], r["vs"][k + 1])
            d = np.linalg.norm(r["pos"]["head"][i0:i1] - c[r["vw"][k]], axis=1)
            out = np.flatnonzero(d > coop.RADIUS_CM)
            tt.append(r["pos_t"][i0 + out[0]] if len(out) else r["ve"][k])
        tt = np.array(tt)
    elif moment == "b":
        tt = r["ve"][:-1]
    else:
        tt = r["vs"][1:] - 1e-6
    j = np.searchsorted(q["poke_t"], tt, side="right") - 1
    ok = j >= 0
    return q["poke_w"][j[ok]] - 1, r["vw"][1:][ok] - 1

def sk_acc(x, y):
    if len(np.unique(y)) < 2 or np.bincount(y).max() < 5:
        return np.nan, None
    X = np.eye(3)[x]
    skf = StratifiedKFold(5)
    accs, agree = [], []
    for tr, te in skf.split(X, y):
        m = LogisticRegression().fit(X[tr], y[tr])
        accs.append(np.mean(m.predict(X[te]) == y[te]))
        agree.append(np.array_equal(m.predict(np.eye(3)), modal_pred(x[tr], y[tr])))
    return np.mean(accs), np.mean(agree)

def modal_pred(x, y):
    cnt = np.zeros((3, 3)); np.add.at(cnt, (x, y), 1)
    overall = np.argmax(np.bincount(y, minlength=3))
    return np.array([np.argmax(cnt[k]) if cnt[k].sum() else overall for k in range(3)])

def null_acc(x, y, rng, n=1000):
    """Mean accuracy over n independent shuffles of x and y, modal-class predictor, stratified folds."""
    N = len(y)
    xs = x[np.argsort(rng.random((n, N)), axis=1)]
    ys = y[np.argsort(rng.random((n, N)), axis=1)]
    oh = ys[:, :, None] == np.arange(3)
    fold = ((np.cumsum(oh, axis=1) - 1) * oh).sum(2) % 5
    cnt = np.bincount((np.arange(n)[:, None] * 45 + fold * 9 + xs * 3 + ys).ravel(), minlength=n * 45).reshape(n, 5, 3, 3)
    tot = cnt.sum(1)
    correct = np.zeros(n)
    for f in range(5):
        tr = tot - cnt[:, f]
        ov = np.argmax(tr.sum(1), axis=1)
        pred = np.where(tr.sum(2) > 0, np.argmax(tr, 2), ov[:, None])
        correct += np.take_along_axis(cnt[:, f], pred[:, :, None], 2)[:, :, 0].sum(1)
    return (correct / N).mean()

rng = np.random.default_rng(18451)
J = []
agreement = []
for s0 in [None]:
    pass
units = sorted(set((r["subject"], r["genotype"]) for s in S for r in s["rats"].values()))
for cont in ("rc100", "rc50"):
    for moment in ("a", "b", "c"):
        per = collections.defaultdict(list)
        for s in S:
            if s["contingency"] != cont:
                continue
            for side, r in s["rats"].items():
                per[r["subject"]].append(samples(s, side, moment))
        for subj, g in units:
            x = np.concatenate([a for a, b in per[subj]]); y = np.concatenate([b for a, b in per[subj]])
            acc, ag = sk_acc(x, y)
            agreement.append(ag)
            J.append(dict(subject=subj, genotype=g, contingency=cont, moment=moment, pooling="p1", n=len(y), real=acc, shuffled=null_acc(x, y, rng)))
            accs, nulls = [], []
            for a, b in per[subj]:
                if len(b) >= 10 and len(np.unique(b)) >= 2 and np.bincount(b).max() >= 5:
                    acc_s, ag_s = sk_acc(a, b)
                    accs.append(acc_s); nulls.append(null_acc(a, b, rng, n=1000))
            J.append(dict(subject=subj, genotype=g, contingency=cont, moment=moment, pooling="p2", n=len(accs), real=np.nanmean(accs), shuffled=np.mean(nulls),
                          dropped=len(per[subj]) - len(accs)))
J = pd.DataFrame(J)
ag = np.array([a for a in agreement if a is not None])
print(f"modal-class shortcut predicts the same class as scikit-learn for every partner well in {ag.mean():.3f} of real folds (pooled reading)")
T = []
for (cont, moment, pooling), d in J.groupby(["contingency", "moment", "pooling"]):
    row = dict(contingency=cont, moment=moment, pooling=pooling)
    for g in ("WT", "Fmr1"):
        dg = d[d.genotype == g]
        u = stats.mannwhitneyu(dg.real, dg.shuffled, alternative="two-sided")
        row[f"{g}_real"] = dg.real.mean(); row[f"{g}_shuf"] = dg.shuffled.mean(); row[f"{g}_U"] = u.statistic; row[f"{g}_p"] = u.pvalue
    u = stats.mannwhitneyu(d[d.genotype == "WT"].real, d[d.genotype == "Fmr1"].real, alternative="two-sided")
    row["WTvsFmr1_U"] = u.statistic; row["WTvsFmr1_p"] = u.pvalue
    T.append(row)
T1j = pd.DataFrame(T)
print(T1j.round(4).to_string())
print(J[(J.pooling == "p1") & (J.moment == "b")].pivot_table(index=["genotype", "subject"], columns="contingency", values=["real", "shuffled", "n"]).round(3).to_string())
# reading (a) reads the partner's location at the moment the rat's head leaves the well, so it rests on positions;
# the same, pooled per rat (p1), with only the aligned pair sessions
Jal = []
for cont in ("rc100", "rc50"):
    per = collections.defaultdict(list)
    for s in S:
        if s["contingency"] == cont and ("-".join(s["pair"]), cont, s["rank"]) in ALIGNED:
            for side, r in s["rats"].items():
                per[r["subject"]].append(samples(s, side, "a"))
    for subj, g in units:
        if subj not in per:
            continue
        x = np.concatenate([a for a, b in per[subj]]); y = np.concatenate([b for a, b in per[subj]])
        acc, _ = sk_acc(x, y)
        Jal.append(dict(subject=subj, genotype=g, contingency=cont, n=len(y), real=acc, shuffled=null_acc(x, y, rng)))
Jal = pd.DataFrame(Jal)
print("reading (a, p1), aligned sessions only:")
print(Jal.round(3).to_string())
for cont, d in Jal.groupby("contingency"):
    out = {g: stats.mannwhitneyu(d[d.genotype == g].real, d[d.genotype == g].shuffled, alternative="two-sided").pvalue for g in ("WT", "Fmr1")}
    out["WT vs Fmr1"] = stats.mannwhitneyu(d[d.genotype == "WT"].real, d[d.genotype == "Fmr1"].real, alternative="two-sided").pvalue
    print(cont, "Mann-Whitney p, real against shuffled and WT against Fmr1 real:", {k: round(v, 4) for k, v in out.items()})

**Result: not reproduced.** Under no reading does the claim hold. WT real accuracy is not higher than Fmr1 at 100% under any reading (Mann-Whitney p 0.12 to 0.27). Fmr1 real accuracy does not exceed the shuffled one at 100% under any reading (p 0.24 to 0.79). At 50% WT exceeds Fmr1 only under reading (b, p2) (p = 0.021). WT real accuracy exceeds its shuffled accuracy in 11 of 12 readings, and at 50% Fmr1 does under readings (a) and (b) (p 0.007 to 0.031). Reading (a) reads the head positions, which disagree with the pokes in 6 of 10 pairs; in the aligned sessions only (reading a, pooled, 4 rats per genotype) no comparison is significant (p 0.20 to 0.89).

## fig3c-triplets-by-contingency

> Under 100% reward rats mostly alternate between two arms, while under 50% they explore all three arms
> through the optimal 1-2-3 or 1-3-2 triplets, especially WT rats.

In [ ]:
TR = []
for s in S:
    for side, r in s["rats"].items():
        tr = coop.triplets(list(r["vw"]))
        TR.append(dict(unit=r["subject"], genotype=r["genotype"], contingency=s["contingency"], rank=s["rank"], n=len(tr),
                       alt=sum(t[0] == t[2] for t in tr), o1=sum(coop.is_optimal(t, "o1") for t in tr), o2=sum(coop.is_optimal(t, "o2") for t in tr)))
TR = pd.DataFrame(TR)
pooled = TR.groupby(["contingency", "genotype", "unit"])[["n", "alt", "o1", "o2"]].sum()
for k in ("alt", "o1", "o2"):
    pooled[f"f_{k}"] = pooled[k] / pooled.n
print(pooled.round(3).to_string())
med = pooled.groupby(["contingency", "genotype"])[["f_alt", "f_o1", "f_o2"]].median()
print(med.round(3).to_string())
for o in ("o1", "o2"):
    c1 = med.loc[("rc100", "WT"), "f_alt"] > 0.5 and med.loc[("rc100", "Fmr1"), "f_alt"] > 0.5
    c2 = med.loc[("rc50", "WT"), f"f_{o}"] > med.loc[("rc100", "WT"), f"f_{o}"]
    c3 = med.loc[("rc50", "WT"), f"f_{o}"] > med.loc[("rc50", "Fmr1"), f"f_{o}"]
    print(f"reading {o}: alternation > 0.5 at 100% in both genotypes {c1}; WT optimal 50% > 100% {c2}; WT > Fmr1 optimal at 50% {c3}")

**Result: not reproduced.** At 50% the WT median fraction of optimal triplets exceeds its 100% value (o1 0.807 against 0.468, o2 0.269 against 0.171) and the Fmr1 median at 50% (0.807 against 0.687; 0.269 against 0.259), under both readings. At 100% the WT median alternation fraction is 0.532 but the Fmr1 median is 0.369, below 0.5 under both readings (the alternation fraction is the same under o1 and o2), so the Fmr1 rats do not predominantly alternate; 4 of 10 Fmr1 rats are above 0.5. The criterion requires all three parts and names no partial outcome.

## fig3d-entropy-wt-lower

> The entropy of WT rats' choice sequences is lower than that of Fmr1 rats, a genotype effect under both
> contingencies.

Sample entropy of each rat's session sequence of visited wells, m = 2 and m = 3, r = 0.2 x SD. A
session with no template match of length m + 1 has undefined sample entropy and is dropped (counted).

In [ ]:
EN = []
for s in S:
    for side, r in s["rats"].items():
        x = np.asarray(r["vw"], float)
        sd = x.std() if len(x) > 1 else 0
        row = dict(unit=r["subject"], genotype=r["genotype"], contingency=s["contingency"], rank=s["rank"])
        for m in (2, 3):
            row[f"se_m{m}"] = coop.sample_entropy(x, m, 0.2 * sd) if sd > 0 else np.nan
        EN.append(row)
EN = pd.DataFrame(EN)
print("undefined (dropped) values:", EN[["se_m2", "se_m3"]].isna().sum().to_dict(), "of", len(EN))
t3d = pd.DataFrame([genotype_tests(EN, f"se_m{m}", c) for m in (2, 3) for c in ("rc100", "rc50")])
print(t3d.round(5).to_string())
for m in (2, 3):
    for c in ("rc100", "rc50"):
        for g in ("WT", "Fmr1"):
            d = EN[(EN.contingency == c) & (EN.genotype == g)].dropna(subset=[f"se_m{m}"])
            lr = stats.linregress(d["rank"], d[f"se_m{m}"])
            print(f"m={m} {c} {g}: slope over sessions {lr.slope:.4f} per session (p = {lr.pvalue:.3g})")
print("range:", EN.se_m2.min().round(3), "to", EN.se_m2.max().round(3), "(m=2);", EN.se_m3.min().round(3), "to", EN.se_m3.max().round(3), "(m=3)")

**Result: partially reproduced.** At 50% WT entropy is lower than Fmr1 under every reading (m = 2: 0.250 against 0.380, reading A p below 0.0001, B p = 0.035; m = 3: 0.240 against 0.365, A p below 0.0001, B p = 0.037). At 100% the genotype effect is not significant under any reading (A p = 0.22 and 0.059, B p = 0.59 and 0.33). The genotype x session terms of reading A are not significant (p 0.24 to 0.74), as the paper reports. Entropy falls over sessions in both genotypes (slopes -0.019 to -0.0025 per session).

## fig3e-exploration-efficiency-wt-higher

> WT rats have higher exploration efficiency (fraction of optimal choice triplets) than Fmr1 rats,
> strongly in the 50% condition and modestly in the 100% condition.

In [ ]:
TR["eff_o1"] = TR.o1 / TR.n.replace(0, np.nan)
TR["eff_o2"] = TR.o2 / TR.n.replace(0, np.nan)
print("rat sessions with no triplet (dropped):", int((TR.n == 0).sum()))
rows3e = []
for o in ("o1", "o2"):
    rows3e.append(dict(genotype_tests(TR, f"eff_{o}", "rc100"), sessions=21))
    for n in (33, 35):
        rows3e.append(dict(genotype_tests(TR[(TR.contingency == "rc100") | (TR["rank"] <= n)], f"eff_{o}", "rc50"), sessions=n))
t3e = pd.DataFrame(rows3e)
print(t3e.round(6).to_string())

**Result: not reproduced.** The criterion needs a WT-higher genotype effect at 50% under every reading. With only 1-2-3 and 1-3-2 counted (o2) there is no difference at 50% (0.273 against 0.262, reading A p = 0.50 and 0.76, and reading B does not find one either). With every triplet visiting all three wells counted (o1), WT efficiency is higher (0.773 against 0.692 over 35 sessions) and reading A finds it (p below 0.0001), reading B does not (p = 0.086 and 0.10). At 100% no reading finds an effect (A p = 0.058 and 0.34).

## figs2-jaccard-similarity

> Within pairs, choice sequences are similar for both genotypes under 100% reward, but under 50%
> reward WT pairs have more similar choice sequences than Fmr1 pairs; in the sensory-deprived condition
> similarity falls in both genotypes alike.

Every session of one rat against every session of its partner (Methods, Eq. 3). Sets: (a) distinct
transitions, (b) distinct nonoverlapping triplets, (c) distinct overlapping triplets. Summaries: (i)
all entries, (ii) lower triangle with the diagonal, (iii) diagonal. Two-sided Mann-Whitney U on the 5
and 5 pair values (the claims file's stand-in). The sensory-deprived sessions are not in the dataset.

In [ ]:
def sets(seq, kind):
    seq = list(seq)
    if kind == "a":
        return set(zip(seq[:-1], seq[1:]))
    if kind == "b":
        return set(coop.triplets(seq))
    return set(zip(seq[:-2], seq[1:-1], seq[2:]))

def jac(a, b):
    u = a | b
    return len(a & b) / len(u) if u else np.nan

JA = []
for pair in sorted(set(s["pair"] for s in S)):
    for cont in ("rc100", "rc50"):
        ss = sorted([s for s in S if s["pair"] == pair and s["contingency"] == cont], key=lambda s: s["rank"])
        g = ss[0]["genotype"]
        r1 = [{r["subject"]: r for r in s["rats"].values()}[pair[0]]["vw"] for s in ss]
        r2 = [{r["subject"]: r for r in s["rats"].values()}[pair[1]]["vw"] for s in ss]
        for kind in ("a", "b", "c"):
            A = [sets(x, kind) for x in r1]; B = [sets(x, kind) for x in r2]
            M = np.array([[jac(a, b) for a in A] for b in B])  # rows: rat 2 sessions, columns: rat 1
            JA.append(dict(pair="-".join(pair), genotype=g, contingency=cont, sets=kind, i=np.nanmean(M),
                           ii=np.nanmean(M[np.tril_indices(len(M))]), iii=np.nanmean(np.diag(M))))
JA = pd.DataFrame(JA)
rows = []
for (cont, kind), d in JA.groupby(["contingency", "sets"]):
    for summ in ("i", "ii", "iii"):
        w = d[d.genotype == "WT"][summ]; f = d[d.genotype == "Fmr1"][summ]
        u = stats.mannwhitneyu(w, f, alternative="two-sided")
        rows.append(dict(contingency=cont, sets=kind, summary=summ, WT=w.mean(), Fmr1=f.mean(), U=u.statistic, p=u.pvalue))
TJ = pd.DataFrame(rows)
print(TJ.round(4).to_string())

**Result: not reproduced.** At 50% WT similarity does not exceed Fmr1 under any reading. Under the transition and overlapping-triplet sets it is lower (for example 0.750 against 0.893, U = 2, p = 0.032, transitions, all entries), and under the nonoverlapping-triplet sets the difference is not significant (p 0.22 to 0.31). At 100% similarity is 0.35 to 0.86 in both genotypes with no difference (p 0.22 to 1.0). The sensory-deprived sessions are not in the dandiset.

## Timing at the wells (Figure 4B to 4D)

For each of the dataset's matches, each rat's visit is its latest visit at the matched well that
began before the match. Arrival: (a) the first poke of that visit; (b) the first frame of the run of
frames in which the head is within 40 px of the well centre that contains the match time.
Departure: (a) the last poke of the visit; (b) the last frame of that run. Dwell, per visit (all
visits): (a) first to last poke; (b) the length of each run of frames within 40 px of a well.
40 px is taken in each video's own pixels (data check: 6.4 cm for most rats, 10 cm for FXM105,
FXM107, FXM108 and FXM109); the readings with 6.4 cm for every rat, as in the first round, are
shown as b64.

In [ ]:
def head_lags(s, radius):
    """Per match, arrival and departure lags from head occupancy; dwell per run. radius(r) in cm."""
    per_rat, dwell = {}, {}
    for side, r in s["rats"].items():
        c = coop.well_centres(r["pos"]["head"], r["pos_t"], r["poke_t"], r["poke_w"])
        occ = coop.occupancy(r["pos"]["head"], c, radius(r))
        per_rat[side] = occ
        b = np.r_[0, occ, 0]
        starts = np.flatnonzero((b[1:-1] != 0) & (b[1:-1] != b[:-2]))
        ends = np.flatnonzero((b[1:-1] != 0) & (b[1:-1] != b[2:]))
        dwell[side] = r["pos_t"][ends] - r["pos_t"][starts]
    arr, dep = [], []
    for t, w in zip(s["match_t"], s["match_w"]):
        tb, db = {}, {}
        for side, occ in per_rat.items():
            r = s["rats"][side]
            i = min(np.searchsorted(r["pos_t"], t), len(occ) - 1)
            if occ[i] == w:
                i0 = i
                while i0 > 0 and occ[i0 - 1] == w:
                    i0 -= 1
                i1 = i
                while i1 < len(occ) - 1 and occ[i1 + 1] == w:
                    i1 += 1
                tb[side], db[side] = r["pos_t"][i0], r["pos_t"][i1]
        if len(tb) == 2:
            arr.append(abs(tb["left"] - tb["right"])); dep.append(abs(db["left"] - db["right"]))
    return np.array(arr), np.array(dep), dwell

def mean_or_nan(x):
    return np.mean(x) if len(x) else np.nan

def median_or_nan(x):
    return np.median(x) if len(x) else np.nan

TM, DW = [], []
for s in S:
    hb = head_lags(s, lambda r: 40 * r["cm_per_px"])
    h64 = head_lags(s, lambda r: coop.RADIUS_CM)
    for side, r in s["rats"].items():
        dw_a = r["ve"] - r["vs"]
        DW.append(dict(unit=r["subject"], genotype=r["genotype"], contingency=s["contingency"], rank=s["rank"],
                       dwell_a=mean_or_nan(dw_a), dwell_b=mean_or_nan(hb[2][side]), dwell_b64=mean_or_nan(h64[2][side])))
    arr = {"a": []}; dep = {"a": []}
    for t, w in zip(s["match_t"], s["match_w"]):
        ta, da = {}, {}
        for side, r in s["rats"].items():
            k = np.flatnonzero((r["vw"] == w) & (r["vs"] <= t + 0.01))
            if len(k) == 0:
                break
            k = k[-1]
            ta[side], da[side] = r["vs"][k], r["ve"][k]
        if len(ta) == 2:
            arr["a"].append(abs(ta["left"] - ta["right"])); dep["a"].append(abs(da["left"] - da["right"]))
    TM.append(dict(unit="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], matches=len(s["match_t"]),
                   arr_a=mean_or_nan(arr["a"]), dep_a=mean_or_nan(dep["a"]),
                   arr_a_med=median_or_nan(arr["a"]), dep_a_med=median_or_nan(dep["a"]), arr_b_med=median_or_nan(hb[0]), dep_b_med=median_or_nan(hb[1]),
                   arr_b=mean_or_nan(hb[0]), dep_b=mean_or_nan(hb[1]), arr_b64=mean_or_nan(h64[0]), dep_b64=mean_or_nan(h64[1]),
                   arr_b_max=np.max(hb[0]) if len(hb[0]) else np.nan, dep_b_max=np.max(hb[1]) if len(hb[1]) else np.nan,
                   arr_b64_max=np.max(h64[0]) if len(h64[0]) else np.nan, dep_b64_max=np.max(h64[1]) if len(h64[1]) else np.nan,
                   n_a=len(arr["a"]), n_b=len(hb[0]), n_b64=len(h64[0])))
TM = pd.DataFrame(TM); DW = pd.DataFrame(DW)
print(f"matches with both arrivals by pokes: {TM.n_a.sum()} of {TM.matches.sum()}; by head position: {TM.n_b.sum()} (40 px), {TM.n_b64.sum()} (6.4 cm)")
print(TM.groupby(["contingency", "genotype"])[["arr_a", "arr_b", "arr_b64", "dep_a", "dep_b", "dep_b64"]].describe().T.round(2).to_string())
print(DW.groupby(["contingency", "genotype"])[["dwell_a", "dwell_b", "dwell_b64"]].describe().T.round(2).to_string())

## fig4b-arrival-lag-wt-shorter

> During match events WT pairs arrive at the wells with shorter lags between the two rats than Fmr1
> pairs, under both contingencies; in WT pairs the lag falls over 100% training.

In [ ]:
t4b = pd.DataFrame([genotype_tests(TM, dv, c) for dv in ("arr_a", "arr_b", "arr_b64") for c in ("rc100", "rc50")])
print(t4b.round(5).to_string())
for dv in ("arr_a", "arr_b"):
    d = TM[(TM.contingency == "rc100") & (TM.genotype == "WT")].dropna(subset=[dv])
    lr = stats.linregress(d["rank"], d[dv])
    print(f"{dv}: WT lag over 100% sessions slope {lr.slope:.3f} s per session (p = {lr.pvalue:.3g})")
    print("  per pair means:", per_unit_means(TM, dv, "rc100"), per_unit_means(TM, dv, "rc50"))
TM["aligned"] = [(u, c, k) in ALIGNED for u, c, k in zip(TM.unit, TM.contingency, TM["rank"])]
print("reading (b) per session, FXM101-FXM110 at 100%: mean and largest arrival and departure lag (s)")
print(TM[(TM.unit == "FXM101-FXM110") & (TM.contingency == "rc100")][["rank", "n_b", "arr_b", "arr_b_max", "dep_b", "dep_b_max"]].round(1).to_string(index=False))
big = TM[(TM.arr_b_max > 100) | (TM.dep_b_max > 100)]
print("pair sessions with a reading-(b) lag over 100 s:", big.groupby(["unit", "contingency"]).size().to_dict())
print("variants of reading (b): sessions with a lag over 100 s removed; only aligned sessions (both rats' pokes and heads agree, data check):")
rows = []
for lab, d in (("lag over 100 s removed", TM[~((TM.arr_b_max > 100) | (TM.dep_b_max > 100))]), ("aligned sessions only", TM[TM.aligned])):
    for dv in ("arr_b", "dep_b"):
        for c in ("rc100", "rc50"):
            dd = d[d.contingency == c]
            rows.append(dict(variant=lab, dv=dv, contingency=c, pair_sessions=dd[dv].notna().sum(), pairs_WT=dd[dd.genotype == "WT"].unit.nunique(), pairs_Fmr1=dd[dd.genotype == "Fmr1"].unit.nunique(),
                             **{k: v for k, v in genotype_tests(d, dv, c).items() if k in ("WT", "Fmr1", "A_F", "A_df", "A_p", "B_p")}))
t4bv = pd.DataFrame(rows)
print(t4bv.round(5).to_string())
print("per pair session median lag in place of the mean, all sessions, no exclusion:")
print(pd.DataFrame([genotype_tests(TM, dv, c) for dv in ("arr_a_med", "arr_b_med") for c in ("rc100", "rc50")]).round(5).to_string())
print("reading (a), by pokes, in the aligned sessions only:")
print(pd.DataFrame([genotype_tests(TM[TM.aligned], dv, c) for dv in ("arr_a", "dep_a") for c in ("rc100", "rc50")]).round(5).to_string())

**Result: not reproduced (changed from blocked after the third verification round).** By pokes (a), WT lags are shorter at 100% (16.6 against 23.3 s, A p = 0.018, B p = 0.19) but not at 50% (12.1 against 13.5 s, A p = 0.43, B p = 0.73), so reading (a) fails. By head position (b), with 40 px in each video's own pixels, WT lags are shorter at 50% (4.0 against 6.0 s, A p below 0.0001, B p = 0.021) but not at 100% (15.4 against 6.3 s, A p = 0.39, B p = 0.46), so reading (b) fails too. The 100% WT mean comes from one session: FXM101-FXM110 session 15 has a mean lag of 803 s and a largest lag of 1,378 s, where its other sessions have means of 2 to 9 s. Four pair sessions have a reading-(b) lag over 100 s; with them removed, WT lags are shorter in both contingencies (4.7 against 6.3 s at 100%, A p = 0.0026, B p = 0.015; 50% as above), reading (b) passes, and the readings would disagree, which gave the blocked outcome of the first two rounds. The 100 s threshold was chosen after seeing session 15 and is not a physical bound: by pokes, session mean lags reach 184.5 s (Fmr1, 100%) and 208.6 s (WT, 50%) (cell above). With the per-session median lag in place of the mean, a reading that needs no exclusion, reading (b) still fails at 100% (13.7 against 4.5 s, A p = 0.39, B p = 0.42). With every session, as the claims file measures the lag, both readings fail, so they agree and the outcome is not reproduced. Session 15 is probably a tracking fault, so the failure of reading (b) at 100% rests on it, but reading (a) does not use head positions and fails at 50% whatever is done with that session. The median by pokes gives WT shorter in both contingencies under reading A (p = 0.033 and below 0.0001) and not under reading B (p = 0.15 and 0.14). The head positions agree with the pokes in only 4 of 10 pairs (data check); in the 189 aligned pair sessions (2 pairs per genotype) reading (b) gives WT shorter at 50% (A p = 0.0033, B p = 0.34) and not at 100% (A p = 0.59), and reading (a) gives A p = 0.0009 at 50% and 0.47 at 100%. The WT lag falls over the 100% sessions by pokes (-1.0 s per session, p = 0.00015) and not by head position (p = 0.59). With 6.4 cm for every rat (b64), the first round's radius, the results are the same (100% A p = 0.37, 50% A p below 0.0001).

## fig4c-departure-lag-wt-shorter

> After match events WT rats depart from the wells with shorter lags between partners than Fmr1 rats,
> under both contingencies.

In [ ]:
rows = []
for dv in ("dep_a", "dep_b", "dep_b64"):
    rows.append(dict(genotype_tests(TM, dv, "rc100"), sessions=21))
    for n in (33, 35):
        rows.append(dict(genotype_tests(TM[(TM.contingency == "rc100") | (TM["rank"] <= n)], dv, "rc50"), sessions=n))
t4c = pd.DataFrame(rows)
print(t4c.round(5).to_string())
print("variants of reading (b) (from the fig4b cell, 35 sessions at 50%):")
print(t4bv[t4bv.dv == "dep_b"].round(5).to_string())
print("per pair session median lag in place of the mean, all sessions, no exclusion:")
rows = []
for dv in ("dep_a_med", "dep_b_med"):
    rows.append(dict(genotype_tests(TM, dv, "rc100"), sessions=21))
    for n in (33, 35):
        rows.append(dict(genotype_tests(TM[(TM.contingency == "rc100") | (TM["rank"] <= n)], dv, "rc50"), sessions=n))
print(pd.DataFrame(rows).round(5).to_string())

**Result: not reproduced (changed from blocked after the third verification round).** By pokes (a) there is no genotype effect in either contingency (A p = 0.17 at 100%, 0.64 and 0.71 at 50%; B p = 0.56, 0.81 and 0.85). By head position (b), with all sessions, WT pairs depart with shorter lags at 50% (4.7 against 8.0 s, A p below 0.0001, B p = 0.084 and 0.086) but not at 100% (A p = 0.67, B p = 0.84), so both readings fail. With the four pair sessions that have a lag over 100 s removed (fig4b cell), WT lags are shorter in both contingencies under reading A (4.1 against 9.3 s at 100%, p = 0.0007; 50% p below 0.0001), though not under reading B (p = 0.19 and 0.084); reading (b) then passes under A where reading (a) fails, which gave the blocked outcome of the first two rounds. That threshold was chosen after seeing the outlier session (fig4b). With the per-session median lag, which needs no exclusion, reading (b) fails at 100% (A p = 0.69, B p = 0.84). With every session, as the claims file measures the lag, both readings fail, so they agree and the outcome is not reproduced. The median by pokes gives WT shorter under reading A (p = 0.011 at 100%, 0.033 and 0.029 at 50%) and not under reading B (p = 0.25, 0.37 and 0.37). In the aligned sessions only, reading (b) gives 50% A p = 0.0004 and 100% A p = 0.60, and reading (a) gives 100% A p = 0.39 and 50% A p = 0.76.

## fig4d-dwell-time

> Fmr1 rats spend longer at reward wells per visit than WT rats (Results), though the legend's genotype
> effects are not significant (p = 0.25 at 100%, p = 0.063 at 50%).

In [ ]:
t4d = pd.DataFrame([genotype_tests(DW, dv, c) for dv in ("dwell_a", "dwell_b", "dwell_b64") for c in ("rc100", "rc50")])
print(t4d.round(5).to_string())
print("per rat mean dwell (a), 50%:", per_unit_means(DW, "dwell_a", "rc50"))
DW["pair"] = DW.unit.map({r["subject"]: "-".join(s["pair"]) for s in S for r in s["rats"].values()})
DWal = DW[[(p, c, k) in ALIGNED for p, c, k in zip(DW.pair, DW.contingency, DW["rank"])]]
print("reading (b), aligned sessions only:")
print(pd.DataFrame([genotype_tests(DWal, "dwell_b", c) for c in ("rc100", "rc50")]).round(5).to_string())

**Result: reproduced (resting on an assumption).** By pokes (a), Fmr1 means are higher than WT means in both contingencies (33.4 against 30.1 s at 100%, 18.3 against 16.9 s at 50%) and the genotype effect of reading A is not significant (p = 0.21 and 0.35), as the legend reports. By head position (b), with 40 px in each video's own pixels, Fmr1 is higher at 100% (5.25 against 5.12 s, p = 0.62) and WT at 50% (5.54 against 5.37 s, p = 0.45), so the readings disagree at 50%. Reading (a) is adopted: the pokes are the task's own record of a visit, while the head positions disagree with the pokes in 6 of 10 pairs (data check). In the aligned sessions, reading (b) agrees with (a) (Fmr1 6.95 against 6.53 s at 100%, 6.62 against 6.56 s at 50%; A p = 0.15 and 0.97). With 6.4 cm for every rat (b64), Fmr1 is higher in both (5.25 against 5.04 s at 100%, 5.35 against 5.34 s at 50%, a difference of 0.009 s). The criteria of fig4b-arrival-lag-wt-shorter and fig4c-departure-lag-wt-shorter block a claim when the readings disagree, so neither reading is preferred there; this criterion has no such clause, so a reading is adopted. In fig4b and fig4c both readings fail with every session, so no preference is needed there.

## Cross-correlation of well arrivals (Figure 4E to 4I)

Occupancy: the nearest well within 40 px of the head (in each video's own pixels, data check), else
null; 1 Hz by (f) the frame at each second or (m) the modal value in each second; arrivals where the
well differs from the previous second, with (y) or without (n) a return to the same well after null
counted. Positive lag: rat 2 (the second subject of the pair, alphabetically) follows rat 1. Lags -20
to +20 s. The shuffle is a circular shift of rat 2's arrival series by a uniform random amount, 20
shifts per session averaged. Variants of reading fy: fy64, 6.4 cm for every rat (the first round's
radius); fy restricted to the aligned sessions of the data check (fy_aligned).

In [ ]:
def xc_session(s, mode, null_return, rng, radius=None):
    rats = {r["subject"]: r for r in s["rats"].values()}
    ser = []
    for subj in s["pair"]:
        r = rats[subj]
        c = coop.well_centres(r["pos"]["head"], r["pos_t"], r["poke_t"], r["poke_w"])
        occ = coop.occupancy(r["pos"]["head"], c, 40 * r["cm_per_px"] if radius is None else radius)
        ser.append(coop.arrivals(coop.downsample_occ(occ, r["pos_t"], s["t0"], s["t1"], "frame" if mode == "f" else "modal"), null_return))
    n = min(len(ser[0]), len(ser[1]))
    a1, a2 = ser[0][:n], ser[1][:n]
    lags, c = coop.xcorr(a1, a2)
    sh = np.mean([coop.xcorr(a1, np.roll(a2, int(k)), 20)[1] for k in rng.integers(1, n, 20)], axis=0)
    return lags, c, sh, a1.sum(), a2.sum()

rng = np.random.default_rng(18452)
XC, PK = [], []
for mode, nr, radius in (("f", "y", None), ("f", "n", None), ("m", "y", None), ("m", "n", None), ("f", "y64", coop.RADIUS_CM)):
        for s in S:
            lags, c, sh, n1, n2 = xc_session(s, mode, nr.startswith("y"), rng, radius)
            k = np.abs(lags) <= 20
            for L, v, vs in zip(lags[k], c[k], sh):
                XC.append(dict(reading=mode + nr, unit="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], lag=int(L), coef=v, shuf=vs))
            ok = np.isfinite(c)
            row = dict(reading=mode + nr, unit="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], arrivals1=n1, arrivals2=n2)
            if ok.any() and np.nanmax(c) > 0:
                kk = k & ok
                row.update(peak20=np.nanmax(c[kk]), lag20=abs(lags[kk][np.nanargmax(c[kk])]), peakall=np.nanmax(c), lagall=abs(lags[np.nanargmax(c)]))
            PK.append(row)
XC = pd.DataFrame(XC); PK = pd.DataFrame(PK)
al = [(u, c, k) in ALIGNED for u, c, k in zip(XC.unit, XC.contingency, XC["rank"])]
XC = pd.concat([XC, XC[(XC.reading == "fy") & np.array(al)].assign(reading="fy_aligned")], ignore_index=True)
al = [(u, c, k) in ALIGNED for u, c, k in zip(PK.unit, PK.contingency, PK["rank"])]
PK = pd.concat([PK, PK[(PK.reading == "fy") & np.array(al)].assign(reading="fy_aligned")], ignore_index=True)
print("sessions without arrivals in a rat (no correlation):", PK.groupby("reading").peak20.apply(lambda x: int(x.isna().sum())).to_dict())
print("arrivals per rat and session (reading fy):", PK[PK.reading == "fy"][["arrivals1", "arrivals2"]].describe().round(1).to_string())

## fig4eg-cross-correlation-wt-higher

> Cross-correlation of the two rats' well-arrival times has peaks above shuffled controls in both
> genotypes, but correlation is higher in WT pairs and only WT pairs show prominent peaks near zero lag.

Linear mixed model coef ~ genotype x lag (lag categorical, sum coding) with a random intercept per
pair; Wald tests of the genotype term and of the genotype x lag terms.

In [ ]:
import statsmodels.formula.api as smf

def lmm_genotype_lag(d):
    d = d.dropna(subset=["coef"]).copy()
    fit = smf.mixedlm("coef ~ C(genotype, Sum) * C(lag, Sum)", d, groups=d["unit"]).fit(reml=True)
    names = fit.fe_params.index
    out = {}
    for term, sel in (("genotype", [i for i, n in enumerate(names) if n.startswith("C(genotype, Sum)") and ":" not in n]),
                      ("genotype x lag", [i for i, n in enumerate(names) if n.startswith("C(genotype, Sum)") and ":" in n])):
        L = np.zeros((len(sel), len(fit.params)))
        for j, i in enumerate(sel):
            L[j, i] = 1
        w = fit.wald_test(L, scalar=True)
        out[term] = (float(w.statistic) / len(sel), len(sel), int(fit.nobs - len(names)), float(w.pvalue))
    return out

rows = []
for reading in ("fy", "fn", "my", "mn", "fy64", "fy_aligned"):
    for c in ("rc100", "rc50"):
        d = XC[(XC.reading == reading) & (XC.contingency == c)]
        res = lmm_genotype_lag(d)
        m = d.groupby("genotype").coef.mean()
        z = d[d.lag.abs() <= 1].groupby("genotype").coef.mean()
        sh = d.groupby("genotype").shuf.mean()
        rows.append(dict(reading=reading, contingency=c, WT_mean=m["WT"], Fmr1_mean=m["Fmr1"], WT_shuf=sh["WT"], Fmr1_shuf=sh["Fmr1"],
                         WT_lag0=z["WT"], Fmr1_lag0=z["Fmr1"], F_genotype=res["genotype"][0], df_g=f"{res['genotype'][1]},{res['genotype'][2]}", p_genotype=res["genotype"][3],
                         F_int=res["genotype x lag"][0], df_int=f"{res['genotype x lag'][1]},{res['genotype x lag'][2]}", p_int=res["genotype x lag"][3]))
t4eg = pd.DataFrame(rows)
print(t4eg.round(5).to_string())
prof = XC[XC.reading == "fy"].groupby(["contingency", "genotype", "lag"])[["coef", "shuf"]].mean().unstack(["contingency", "genotype"])
print("mean coefficient by lag, reading fy (observed and shuffled):")
print(prof.round(4).to_string())
print("coefficient range:", XC.coef.min().round(4), "to", XC.coef.max().round(4))

**Result: not reproduced.** The criterion needs a WT-higher genotype effect in both contingencies, and the 50% genotype effect is not significant under any reading (p 0.29 to 0.45; 0.35 with 6.4 cm for every rat). At 100% it is significant only when returns to the same well are not counted (p = 0.023 and 0.030; frame sampling with returns p = 0.18). The rest holds: WT coefficients are higher than Fmr1 at every reading and contingency, and at lags -1 to +1 s (frame sampling, returns counted: 0.078 against 0.057 at 100%, 0.085 against 0.070 at 50%), and the genotype x lag interaction is significant at 50% under every reading (p below 0.0001) and at 100% under every reading over all sessions (p 0.0002 to 0.026; in the aligned sessions only p = 0.17). Near zero lag the WT mean at 100% (0.078) is above the mean shuffled coefficient of WT sessions (0.066); the Fmr1 mean (0.057) is close to its own (0.055). In the aligned sessions only, the genotype effect is not significant in either contingency (p = 0.82 and 0.87).

## fig4h-peak-lag-distribution

> The distribution of lags at peak cross-correlation differs between WT and Fmr1 pairs in the 50%
> condition but not in the 100% condition.

In [ ]:
rows = []
for reading in ("fy", "fn", "my", "mn", "fy64", "fy_aligned"):
    for rng_ in ("20", "all"):
        for c in ("rc100", "rc50"):
            d = PK[(PK.reading == reading) & (PK.contingency == c)].dropna(subset=[f"lag{rng_}"])
            w = d[d.genotype == "WT"][f"lag{rng_}"]; f = d[d.genotype == "Fmr1"][f"lag{rng_}"]
            ks = stats.ks_2samp(w, f)
            rows.append(dict(reading=reading, range=rng_, contingency=c, n_WT=len(w), n_Fmr1=len(f), WT_median=w.median(), Fmr1_median=f.median(), D=ks.statistic, p=ks.pvalue))
t4h = pd.DataFrame(rows)
print(t4h.round(5).to_string())

**Result: not reproduced (resting on an assumption).** Under the reading the Methods describe most directly (the frame at each second, every change into a well an arrival, 40 px in each video's own pixels, lags -20 to +20 s), the distributions differ in neither contingency (KS p = 0.15 at 100%, 0.38 at 50%). With lags -20 to +20 s, the 50% test is not below 0.01 under any reading (p 0.024 to 0.81). Over all lags, the claimed pattern (50% p below 0.01, 100% p at or above 0.05) appears in one reading (frame sampling, returns not counted: p = 0.11 and 0.0026).

## fig4i-peak-correlation-wt-higher

> Session-wise peak cross-correlation coefficients are higher in WT than in Fmr1 pairs under both
> contingencies.

In [ ]:
rows = []
for reading in ("fy", "fn", "my", "mn", "fy64", "fy_aligned"):
    for rng_ in ("20", "all"):
        for c in ("rc100", "rc50"):
            d = PK[PK.reading == reading]
            rows.append(dict(genotype_tests(d, f"peak{rng_}", c), reading=reading, range=rng_))
t4i = pd.DataFrame(rows)
print(t4i.round(5).to_string())

**Result: blocked (methods underspecified).** WT peak coefficients are higher in every reading (for example 0.143 against 0.122 at 100% and 0.148 against 0.136 at 50%, frame sampling, lags to 20 s). Reading A finds the genotype effect in both contingencies in all 8 readings (p 0.011 or below); reading B finds it in both contingencies in 3 of 8 (returns not counted: frame sampling with lags to 20 s, p = 0.007 and 0.044, and all lags, p = 0.003 and 0.031; modal sampling with lags to 20 s, p = 0.010 and 0.044). The ANOVA readings disagree.

## Leaders of match events (Figure 5)

For each of the dataset's matches, the leader is the rat whose visit at the matched well began first
(first poke of the visit; Methods, Assignment of leader and follower). Ties (equal first pokes) are
counted and dropped.

In [ ]:
LEAD = []
for s in S:
    rats = {r["subject"]: r for r in s["rats"].values()}
    for t, w in zip(s["match_t"], s["match_w"]):
        st = {}
        for subj, r in rats.items():
            k = np.flatnonzero((r["vw"] == w) & (r["vs"] <= t + 0.01))
            if len(k):
                st[subj] = r["vs"][k[-1]]
        if len(st) == 2:
            a, b = s["pair"]
            if st[a] == st[b]:
                LEAD.append(dict(pair="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], t=t, leader=None))
            else:
                LEAD.append(dict(pair="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], t=t,
                                 leader=a if st[a] < st[b] else b, lag=abs(st[a] - st[b])))
LEAD = pd.DataFrame(LEAD)
print(f"matches with a leader: {LEAD.leader.notna().sum()} of {len(LEAD)} (ties {LEAD.leader.isna().sum()})")
LEAD = LEAD[LEAD.leader.notna()].sort_values(["pair", "contingency", "rank", "t"])

## fig5d-leading-near-half

> Leadership is not fixed; on average rats of both genotypes lead about half of the match events, with
> no genotype difference in the mean probability of leading estimated by a state-space model.

Per rat and contingency, the binary series of leading over all its match events, sessions in order,
fitted with the Smith et al. (2004) model (`coop.smith_ssm`, b0 = 0); the rat's mean p_k.

In [ ]:
SSM = []
for (pair, cont), d in LEAD.groupby(["pair", "contingency"]):
    g = d.genotype.iloc[0]
    for subj in pair.split("-"):
        n = (d.leader == subj).astype(int).values
        pk, ve = coop.smith_ssm(n)
        SSM.append(dict(subject=subj, pair=pair, genotype=g, contingency=cont, events=len(n), raw=n.mean(), ssm_mean=pk.mean(), ssm_min=pk.min(), ssm_max=pk.max(), ve=ve))
SSM = pd.DataFrame(SSM)
print(SSM.round(4).to_string())
for c in ("rc100", "rc50"):
    d = SSM[SSM.contingency == c]
    u = stats.mannwhitneyu(d[d.genotype == "WT"].ssm_mean, d[d.genotype == "Fmr1"].ssm_mean, alternative="two-sided")
    print(f"{c}: mean of rat means WT {d[d.genotype == 'WT'].ssm_mean.mean():.4f}, Fmr1 {d[d.genotype == 'Fmr1'].ssm_mean.mean():.4f}; Mann-Whitney U = {u.statistic:.1f}, p = {u.pvalue:.4f}; rat means {d.ssm_mean.min():.3f} to {d.ssm_mean.max():.3f}")

**Result: reproduced.** The genotype means of the state-space mean probability of leading are 0.500 in both genotypes and contingencies, and the Mann-Whitney tests give U = 50, p = 1.0. This follows by construction in same-genotype pairs; the rat means range from 0.290 to 0.710 at 100% and 0.315 to 0.685 at 50%.

## fig5e-leading-difference-50

> The within-pair difference in the proportion of leading events does not differ by genotype at 100%,
> but shows a small significant genotype effect at 50%, present at the start of training and gone later
> in training (Results).

In [ ]:
LD = []
for (pair, cont, rank), d in LEAD.groupby(["pair", "contingency", "rank"]):
    a = pair.split("-")[0]
    pa = (d.leader == a).mean()
    LD.append(dict(unit=pair, genotype=d.genotype.iloc[0], contingency=cont, rank=rank, absdiff=abs(2 * pa - 1), signed=2 * pa - 1))
LD = pd.DataFrame(LD)
rows = []
for dv in ("absdiff", "signed"):
    rows.append(dict(genotype_tests(LD, dv, "rc100"), sessions=21))
    for n in (33, 35):
        rows.append(dict(genotype_tests(LD[(LD.contingency == "rc100") | (LD["rank"] <= n)], dv, "rc50"), sessions=n))
t5e = pd.DataFrame(rows)
print(t5e.round(5).to_string())
for lab, sel in (("first five 50% sessions", LD["rank"] <= 5), ("last five analysed 50% sessions", LD["rank"] > 30)):
    d = LD[(LD.contingency == "rc50") & sel]
    print(lab, d.groupby("genotype").absdiff.mean().round(4).to_dict())

**Result: not reproduced.** At 50% there is no genotype effect on the within-pair difference under any reading (absolute difference, A p = 0.70 and 0.73, B p = 0.88 and 0.89; signed difference, A p = 0.061 and 0.070). At 100% there is none either (A p = 0.49), as the paper reports. In the first five 50% sessions the absolute difference is larger in WT pairs (0.397 against 0.253) and in the last five it is similar (0.271 against 0.290), which is the pattern the Results describe, without significance.

## figs3-leadership-history

> A rat's chance of leading on a given match event depends on whether it led on the preceding events.

Blocked as methods_underspecified by the claims file (the analysis is only in the unread supplement).
Shown, not judged: per rat (a) P(lead | led the previous event) against P(lead | followed it), Wilcoxon
signed-rank across rats; (b) a logistic regression of leading on the lead status of each of the
previous 1 to 3 events, pooled over rats.

In [ ]:
import statsmodels.api as sm
HS = []
for (pair, cont), d in LEAD.groupby(["pair", "contingency"]):
    for subj in pair.split("-"):
        n = (d.leader == subj).astype(int).values
        HS.append(dict(subject=subj, contingency=cont, genotype=d.genotype.iloc[0], p_after_lead=n[1:][n[:-1] == 1].mean(), p_after_follow=n[1:][n[:-1] == 0].mean()))
HS = pd.DataFrame(HS)
for c in ("rc100", "rc50"):
    d = HS[HS.contingency == c]
    w = stats.wilcoxon(d.p_after_lead, d.p_after_follow)
    print(f"{c}: P(lead | led) {d.p_after_lead.mean():.3f}, P(lead | followed) {d.p_after_follow.mean():.3f}, Wilcoxon p = {w.pvalue:.3g} (20 rats)")
    X, Y = [], []
    for (pair, cc), dd in LEAD[LEAD.contingency == c].groupby(["pair", "contingency"]):
        subj = pair.split("-")[0]
        n = (dd.leader == subj).astype(int).values
        for k in range(3, len(n)):
            X.append(n[k - 3:k][::-1]); Y.append(n[k])
    fit = sm.Logit(np.array(Y), sm.add_constant(np.array(X, float))).fit(disp=0)
    print(f"   logistic regression on the previous 1, 2, 3 events: coefficients {np.round(fit.params[1:], 3)}, p {np.array([f'{p:.2g}' for p in fit.pvalues[1:]])}")

**Result: blocked (methods underspecified), by the claims file.** Shown only: at 100% P(lead | led) is 0.473 against P(lead | followed) 0.527 (Wilcoxon p = 0.0012, 20 rats), at 50% 0.497 against 0.503 (p = 0.60); in a logistic regression the lead status two and three events back predicts leading in both contingencies (p below 1e-11).

## fig7b-fmr1-more-asynchronous-matches

> Under 50% reward, a larger proportion of Fmr1 pairs' matches occur in asynchronous trials (rats
> starting at different wells) than WT pairs' matches; at 100% there is no genotype difference.

A match is synchronous when the two rats' previous visits (the visits before the matched visits)
were at corresponding wells, and asynchronous otherwise (Results, the two trial types).

In [ ]:
AS = []
for s in S:
    nsync = nasync = 0
    for t, w in zip(s["match_t"], s["match_w"]):
        prev = []
        for r in s["rats"].values():
            k = np.flatnonzero((r["vw"] == w) & (r["vs"] <= t + 0.01))
            if len(k) and k[-1] > 0:
                prev.append(r["vw"][k[-1] - 1])
        if len(prev) == 2:
            if prev[0] == prev[1]:
                nsync += 1
            else:
                nasync += 1
    AS.append(dict(unit="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], sync=nsync, asyn=nasync,
                   prop_async=nasync / (nsync + nasync) if nsync + nasync else np.nan))
AS = pd.DataFrame(AS)
print(AS.groupby(["contingency", "genotype"])[["sync", "asyn"]].sum().to_string())
t7b = pd.DataFrame([genotype_tests(AS, "prop_async", c) for c in ("rc100", "rc50")])
print(t7b.round(6).to_string())
d = AS[AS.contingency == "rc50"].dropna(subset=["prop_async"])
lmm = smf.mixedlm("prop_async ~ C(genotype, Sum) * C(rank, Sum)", d, groups=d["unit"]).fit(reml=True)
names = list(lmm.fe_params.index)
L = np.zeros((1, len(lmm.params))); L[0, names.index([n for n in names if n.startswith("C(genotype, Sum)") and ":" not in n][0])] = 1
w = lmm.wald_test(L, scalar=True)
print(f"50% mixed model, genotype: F(1,{int(lmm.nobs - len(names))}) = {float(w.statistic):.3f}, p = {float(w.pvalue):.3g}")
print("per pair mean proportion asynchronous:", per_unit_means(AS, "prop_async", "rc100"), per_unit_means(AS, "prop_async", "rc50"))

**Result: not reproduced (resting on an assumption).** At 50% a larger proportion of Fmr1 matches are asynchronous (0.363 against 0.192), and the mixed model with a random intercept per pair finds it (F(1,279) = 4.18, p = 0.041); reading A also does (p below 0.0001). At 100% the Fmr1 proportion is also higher (0.359 against 0.231) and reading A, the test the criterion names, finds the difference (p = 0.00024; reading B p = 0.11), so the absence part fails. The criterion requires both parts and names no partial outcome.

## Follower trials (Figure 8)

The Methods: "In each trial, one rat (the leader) enters an arm first; the other rat (the follower)
then decides whether to choose the matching arm and cooperate" (Partner behavior history analysis;
Figure 8A: "one rat (leader) transitions between the arms first; then the other (follower) decides
whether to cooperate"). Trials are built from the arrivals at new wells (first poke of a visit) of
both rats (`coop.follower_trials`): a trial opens at the next arrival by either rat, which leads, and
closes at the other rat's next arrival, its choice; it cooperates when its new well corresponds to
the leader's current well. The partner's previous choice is the outcome of the partner's most recent
follower trial within the session (across sessions as a variant); Δt is the time from it to the
current choice.

In the first round every arrival by a rat whose partner made the pair's previous arrival was a
follower trial (shown as the variant "arrival sequence"). After a partner's cooperative choice the
rat's next arrival had to leave the matched well, so it was counted as not cooperating by
construction (verifier's report, problem 1). Any rule that builds trials from the sequence of
arrivals ties a trial to the previous one, so each rule is also run on a null: the second rat's
visits circularly shifted by a uniform random amount within the session, 20 shifts per session,
which keeps each rat's own sequence and removes any coupling between the two. A rat's difference
P(coop | partner cooperated) - P(coop | partner did not) on the shifted data is what the rule gives
without any social effect.

In [ ]:
def trials_of(s, shift=0.0):
    a, b = (s["rats"][side] for side in ("left", "right"))
    return coop.follower_trials((a["subject"], a["vw"], a["vs"]), (b["subject"], b["vw"], b["vs"]), shift, s["t0"], s["t1"] - s["t0"])

def arrival_sequence(s, shift=0.0):
    """First round's rule: every arrival by a rat whose partner made the pair's previous arrival.
    The right rat's arrivals can be circularly shifted (the null), as in coop.follower_trials."""
    ev = []
    for side, r in s["rats"].items():
        vs = (r["vs"] - s["t0"] + shift) % (s["t1"] - s["t0"]) + s["t0"] if (shift and side == "right") else r["vs"]
        ev += [(vs[k], r["subject"], r["vw"][k]) for k in range(len(r["vw"]))]
    ev.sort()
    cur, last, out = {}, None, []
    for t, subj, w in ev:
        other = [x for x in s["pair"] if x != subj][0]
        if last == other and other in cur:
            out.append((subj, other, t, int(w == cur[other]), int(cur.get(subj) == cur[other]), np.nan))
        cur[subj] = w
        last = subj
    return out

def trial_table(rule, nshift=0, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for s in S:
        shifts = [0.0] if nshift == 0 else rng.uniform(0, s["t1"] - s["t0"], nshift)
        for k, sh in enumerate(shifts):
            for subj, lead, t, c, imp, lat in (trials_of(s, sh) if rule == "methods" else arrival_sequence(s, sh)):
                out.append((k, subj, lead, s["genotype"], "-".join(s["pair"]), s["contingency"], s["rank"], t, c, imp, lat))
    F = pd.DataFrame(out, columns=["k", "subject", "partner", "genotype", "pair", "contingency", "rank", "t", "coop", "impossible", "latency"])
    F["sess"] = F.pair + "_" + F.contingency + "_" + F["rank"].astype(str)
    return F.sort_values(["k", "pair", "contingency", "rank", "t"]).reset_index(drop=True)

def add_history(F, within=True):
    """Partner's most recent follower choice (and Δt to it), the rat's own previous two choices."""
    keys = ["k", "pair", "contingency", "rank"] if within else ["k", "pair", "contingency"]
    F = F.sort_values(keys + (["t"] if within else ["rank", "t"])).reset_index(drop=True)
    pp, dt, o1, o2 = (np.full(len(F), np.nan) for _ in range(4))
    prev_key, last, own = None, {}, {}
    cols = [F[c].values for c in keys] + [F.subject.values, F.partner.values, F.t.values, F.coop.values]
    for i, row in enumerate(zip(*cols)):
        key, (subj, part, t, c) = row[:len(keys)], row[len(keys):]
        if key != prev_key:
            prev_key, last, own = key, {}, {}
        if part in last:
            pp[i], dt[i] = last[part][0], t - last[part][1] if within else np.nan
        h = own.setdefault(subj, [])
        if h:
            o1[i] = h[-1]
        if len(h) > 1:
            o2[i] = h[-2]
        last[subj] = (c, t)
        h.append(c)
    F["partner_prev"], F["dt"], F["own_prev1"], F["own_prev2"] = pp, dt, o1, o2
    return F

FT = trial_table("methods")
FTw = add_history(FT, True)
FTa = add_history(FT, False)
FTold = add_history(trial_table("old"), True)
NT = add_history(trial_table("methods", nshift=20, seed=18458), True)
NTold = add_history(trial_table("old", nshift=20, seed=18459), True)
print(f"follower trials (Methods rule): {len(FT)}; with a partner history within the session {FTw.partner_prev.notna().sum()}, across sessions {FTa.partner_prev.notna().sum()}")
print(f"arrival-sequence rule (first round): {len(FTold)}; null trials (20 shifts per session): Methods rule {len(NT)}, arrival sequence {len(NTold)}")
print("trials, cooperation rate, fraction in which cooperation is impossible (the leader went to the well matching the follower's own), median leader-to-follower latency (s):")
print(FT.groupby(["contingency", "genotype"]).agg(n=("coop", "size"), coop=("coop", "mean"), impossible=("impossible", "mean"), latency=("latency", "median")).round(3).to_string())

## fig8b-partner-history-cooperation

> After a partner's cooperative choice, WT rats are more likely to cooperate under both contingencies,
> Fmr1 rats only under 100% reward.

In [ ]:
def pr_table(F):
    d = F.dropna(subset=["partner_prev"])
    return d.groupby(["contingency", "genotype", "subject", "k", "partner_prev"]).coop.mean().unstack()

def fig8b_tests(F, N, lab):
    """The criterion's Wilcoxon test per group, the same test on the null, and the observed difference
    against the null difference of the same rat (Wilcoxon on observed minus null mean)."""
    po, pn = pr_table(F).droplevel("k"), pr_table(N)
    dn = (pn[1.0] - pn[0.0]).groupby(["contingency", "genotype", "subject"]).mean()
    rows = []
    for c in ("rc100", "rc50"):
        for g in ("WT", "Fmr1"):
            o = po.loc[(c, g)]
            w = stats.wilcoxon(o[1.0], o[0.0])
            nn = pn.loc[(c, g)].groupby("subject").mean()
            wn = stats.wilcoxon(nn[1.0], nn[0.0])
            ex = (o[1.0] - o[0.0]) - dn.loc[(c, g)]
            we = stats.wilcoxon(ex.dropna())
            rows.append(dict(rule=lab, contingency=c, genotype=g, P_after_coop=o[1.0].mean(), P_after_not=o[0.0].mean(), rats_higher=f"{int((o[1.0] > o[0.0]).sum())}/{len(o)}", p=w.pvalue,
                             null_after_coop=nn[1.0].mean(), null_after_not=nn[0.0].mean(), null_rats_higher=f"{int((nn[1.0] > nn[0.0]).sum())}/{len(nn)}", null_p=wn.pvalue,
                             excess_over_null=ex.mean(), excess_rats_above=f"{int((ex > 0).sum())}/{ex.notna().sum()}", excess_p=we.pvalue))
    return rows

rows = fig8b_tests(FTw, NT, "Methods rule")
rows += fig8b_tests(FTw[FTw.impossible == 0], NT[NT.impossible == 0], "Methods rule, possible trials only")
rows += fig8b_tests(FTold, NTold, "arrival sequence (first round)")
t8b = pd.DataFrame(rows)
print(t8b.round(4).to_string())
print("across sessions (variant), Methods rule, criterion's test:")
for c in ("rc100", "rc50"):
    for g in ("WT", "Fmr1"):
        o = pr_table(FTa).droplevel("k").loc[(c, g)]
        print(f"  {c} {g}: {o[1.0].mean():.3f} against {o[0.0].mean():.3f}, Wilcoxon p = {stats.wilcoxon(o[1.0], o[0.0]).pvalue:.4f}")
print("per rat, Methods rule, observed and null difference:")
po, pn = pr_table(FTw).droplevel("k"), pr_table(NT)
print(pd.DataFrame({"observed": po[1.0] - po[0.0], "null": (pn[1.0] - pn[0.0]).groupby(["contingency", "genotype", "subject"]).mean()}).round(3).unstack(0).to_string())

**Result: blocked (the criterion's test cannot separate a partner effect from how follower trials are built).** With follower trials as the Methods describe them, rats cooperate more after their partner cooperated in every group (WT 0.672 against 0.425 at 100% and 0.682 against 0.375 at 50%, p = 0.002 each; Fmr1 0.528 against 0.390, p = 0.014, and 0.528 against 0.369, p = 0.002), so the absence at 50% in Fmr1 rats fails. But the null, with one rat's visits shifted in time so that the two rats are independent, gives the same pattern in every group (10 of 10 rats higher in each, p = 0.002; for example WT at 100% 0.482 against 0.319). Building trials from the sequence of arrivals ties each trial to the previous one, and the first round's rule gives the opposite bias (null 0.176 against 0.506 for WT at 100%). Against the null, the excess is significant in WT rats (+0.083 at 100%, p = 0.014; +0.069 at 50%, p = 0.014) and not in Fmr1 rats (-0.006, p = 0.92; +0.034, p = 0.16); with the first round's rule the excess pattern differs (significant at 100% in both genotypes, p = 0.002 and 0.020, at 50% in neither). The paper does not say how its trials avoid this dependence, and the criterion's test, which compares the two probabilities directly, is passed or failed by the construction alone.

## fig8d-fmr1-longer-delta-t

> The time between the partner's most recent follower choice and the subject's current follower choice
> is longer in Fmr1 than in WT rats under both contingencies.

In [ ]:
for lab, F in (("Methods rule", FTw), ("arrival sequence (first round)", FTold), ("null, Methods rule", NT)):
  print(lab)
  for c in ("rc100", "rc50"):
    d = F[(F.contingency == c)].dropna(subset=["dt"])
    w = d[d.genotype == "WT"].dt; f = d[d.genotype == "Fmr1"].dt
    u = stats.mannwhitneyu(f, w, alternative="two-sided")
    print(f"  {c}: Fmr1 median {f.median():.2f} s (n = {len(f)}), WT median {w.median():.2f} s (n = {len(w)}); U (Fmr1 vs WT) = {u.statistic:.0f}, p = {u.pvalue:.3g}; range {d.dt.min():.2f} to {d.dt.max():.1f} s, above 60 s: {(d.dt > 60).sum()}")
print(FTw.dropna(subset=["dt"]).groupby(["contingency", "genotype", "subject"]).dt.median().round(2).unstack(0).to_string())

**Result: blocked, as fig8b-partner-history-cooperation, on which it depends.** Δt is longer in Fmr1 than in WT rats in both contingencies with follower trials as the Methods describe them (medians 63.6 against 47.0 s at 100%, p = 1.0e-24; 44.6 against 33.0 s at 50%, p = 2.5e-79) and with the first round's rule (16.3 against 11.4 s, p = 1.5e-77; 12.7 against 9.6 s, p = 4.9e-216). The difference is also present on the shifted null (95.7 against 80.4 s at 100%), so it reflects the slower arrivals of Fmr1 rats (fig1f). The U values (5,407,193 and 40,434,726 under the Methods rule) differ from the paper's, as the trial counts do.

## fig8e-glmm-partner-cooperation

> In a GLMM of cooperation on each follower trial, the partner's previous cooperation has a larger effect
> in WT rats at 50% (beta 0.77) than in Fmr1 rats (beta 0.13), a significant genotype interaction (beta
> 0.64); at 100% both genotypes show an effect (WT 0.36, Fmr1 0.26).

Binomial GLMM (logit), partner_prev x Δt x genotype, own history covariates, random intercepts for rat
and session, fitted by statsmodels' `BinomialBayesMixedGLM` (variational Bayes; pymer4 and R are not
in the environment), on the Methods-rule follower trials. Readings: own history 1 or 2 previous
follower choices; Δt in seconds or z-scored (the seconds reading is fitted in minutes, an equivalent
scaling); genotype reference WT (gives the WT beta and Fmr1 minus WT) or Fmr1 (gives the Fmr1 beta and
WT minus Fmr1). p from the posterior mean over its standard deviation. Checks: the same fixed effects
by maximum likelihood (ordinary logistic regression with standard errors clustered by rat), on the
observed trials and on the null trials of the shifted sessions.

In [ ]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
rows = []
for c in ("rc100", "rc50"):
    for own in (1, 2):
        for scale in ("s", "z"):
            d = FTw[FTw.contingency == c].dropna(subset=["partner_prev", "dt", "own_prev1"] + (["own_prev2"] if own == 2 else [])).copy()
            # seconds: fitted with Δt in minutes, the same model with Δt's coefficient rescaled (variational
            # Bayes does not converge with Δt in seconds); the partner effect at Δt = 0 is unchanged
            d["dtx"] = d.dt / 60 if scale == "s" else (d.dt - d.dt.mean()) / d.dt.std()
            for ref in ("WT", "Fmr1"):
                d["g"] = (d.genotype != ref).astype(int)
                cov = " + own_prev1" + (" + own_prev2" if own == 2 else "")
                m = BinomialBayesMixedGLM.from_formula(f"coop ~ partner_prev * dtx * g{cov}", {"rat": "0 + C(subject)", "session": "0 + C(sess)"}, d)
                fit = m.fit_vb()
                names = list(m.exog_names)
                i, j = names.index("partner_prev"), names.index("partner_prev:g")
                for term, k in (("beta_ref", i), ("interaction_other_minus_ref", j)):
                    z = fit.fe_mean[k] / fit.fe_sd[k]
                    rows.append(dict(contingency=c, own=own, dt=scale, reference=ref, term=term, beta=fit.fe_mean[k], sd=fit.fe_sd[k], p=2 * stats.norm.sf(abs(z)), n=len(d)))
t8e = pd.DataFrame(rows)
print(t8e.round(4).to_string())
print("maximum likelihood check (logistic regression, own history 1, Δt z-scored, SE clustered by rat), observed and null trials:")
rows = []
for lab, F in (("observed", FTw), ("null (shifted)", NT)):
    for c in ("rc100", "rc50"):
        d = F[F.contingency == c].dropna(subset=["partner_prev", "dt", "own_prev1"]).copy()
        d["dtx"] = (d.dt - d.dt.mean()) / d.dt.std()
        for ref in ("WT", "Fmr1"):
            d["g"] = (d.genotype != ref).astype(int)
            fit = smf.glm("coop ~ partner_prev * dtx * g + own_prev1", d, family=sm.families.Binomial()).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.subject)[0]})
            for term, name in (("beta_ref", "partner_prev"), ("interaction_other_minus_ref", "partner_prev:g")):
                rows.append(dict(data=lab, contingency=c, reference=ref, term=term, beta=fit.params[name], se=fit.bse[name], p=fit.pvalues[name], n=len(d)))
print(pd.DataFrame(rows).round(4).to_string())

**Result: blocked, as fig8b-partner-history-cooperation, on which it depends.** With follower trials as the Methods describe them, the partner's previous cooperation raises the log-odds of cooperating in both genotypes and contingencies under every reading (WT 0.51 to 0.89 at 100% and 0.82 to 1.44 at 50%; Fmr1 0.54 to 0.98 and 0.62 to 0.88; all p below 0.0001 by variational Bayes), and at 50% the WT minus Fmr1 term is positive (0.20 to 0.57, p below 0.0001). By maximum likelihood with standard errors clustered by rat, the 50% genotype term is not significant (0.18, p = 0.34). The same model on the shifted null gives partner betas of 0.50 to 0.75 (p below 0.0001), so most of the observed effect is what the trial construction gives without any coupling.

## Asynchronous trials and gaze (Figure 7C to 7H)

An asynchronous trial (Methods, Quantification of peer-directed head orientation events): the two
rats are at distinct, noncorresponding wells (each within a visit, first to last poke); the follower
is the rat that departs first (its visit ends) while the partner is still at its well. The trial runs
from the moment both are at those wells to the follower's last poke there. The outcome is a match when
one of the dataset's matched pokes falls at the leader's well between the follower's departure and the
end of its next visit. Trials shorter than 0.5 s (too few frames for quartiles) are dropped.
Gaze per frame from the processed neck, head and body positions (cm, one video frame for both rats):
bearing angle of Eq. 11 below 40 degrees. Follower-only gaze: the follower is oriented to the leader
and the leader is not oriented to the follower; mutual: both.

In [ ]:
TRI = []
dropped_short = 0
for s in S:
    rats = s["rats"]
    sides = ("left", "right")
    for fs in sides:
        ls = "right" if fs == "left" else "left"
        F, L = rats[fs], rats[ls]
        for k in range(len(F["vw"])):
            wf, sf, ef = F["vw"][k], F["vs"][k], F["ve"][k]
            j = np.flatnonzero((L["vs"] <= ef) & (L["ve"] > ef) & (L["vw"] != wf))
            if len(j) == 0:
                continue
            j = j[0]
            start = max(sf, L["vs"][j])
            if ef - start < 0.5:
                dropped_short += 1
                continue
            wl = L["vw"][j]
            nxt_end = F["ve"][k + 1] if k + 1 < len(F["vw"]) else ef
            match = bool(np.any((s["match_w"] == wl) & (s["match_t"] > ef) & (s["match_t"] <= nxt_end + 0.01)))
            # gaze
            iF0, iF1 = np.searchsorted(F["pos_t"], [start, ef])
            tt = F["pos_t"][iF0:iF1]
            if len(tt) < 4:
                dropped_short += 1
                continue
            iL = np.clip(np.searchsorted(L["pos_t"], tt), 0, len(L["pos_t"]) - 1)
            thF = coop.bearing(F["pos"]["neck"][iF0:iF1], F["pos"]["head"][iF0:iF1], L["pos"]["body"][iL])
            thL = coop.bearing(L["pos"]["neck"][iL], L["pos"]["head"][iL], F["pos"]["body"][iF0:iF1])
            ok = np.isfinite(thF) & np.isfinite(thL)
            fo = (thF < 40) & (thL >= 40)
            mu = (thF < 40) & (thL < 40)
            q = np.minimum((np.arange(len(tt)) * 4) // len(tt), 3)
            row = dict(follower=F["subject"], leader=L["subject"], genotype=s["genotype"], pair="-".join(s["pair"]), contingency=s["contingency"], rank=s["rank"],
                       sess=f"{'-'.join(s['pair'])}_{s['contingency']}_{s['rank']}", start=start, end=ef, duration=ef - start, match=int(match),
                       valid=ok.mean(), gF=fo[ok].mean() if ok.any() else np.nan, gM=mu[ok].mean() if ok.any() else np.nan)
            for qq in range(4):
                m = ok & (q == qq)
                row[f"gF_q{qq + 1}"] = fo[m].mean() if m.any() else np.nan
                row[f"gM_q{qq + 1}"] = mu[m].mean() if m.any() else np.nan
            TRI.append(row)
TRI = pd.DataFrame(TRI)
print(f"asynchronous trials: {len(TRI)}; dropped as shorter than 0.5 s or 4 frames: {dropped_short}")
print(TRI.groupby(["contingency", "genotype"]).agg(n=("match", "size"), match=("match", "mean"), duration=("duration", "median"), gF=("gF", "mean"), gM=("gM", "mean"), valid=("valid", "mean")).round(3).to_string())
print(f"trial duration {TRI.duration.min():.2f} to {TRI.duration.max():.1f} s; frames with both bearings defined {TRI.valid.mean():.3f}")

## fig7c-trial-duration-vs-performance

> Session mean trial duration is negatively correlated with cooperative performance in both genotypes
> (WT R = -0.51, Fmr1 R = -0.40).

Readings of the trial: (a) the asynchronous trial above; (b) the interval between consecutive
matches; (c) the interval between consecutive transitions of the pair. Points: one per pair and
session, or one per rat and session (for (a), the rat as follower; for (c), the rat's own
transitions; (b) is a pair measure). 100% and 50% sessions combined.

In [ ]:
perf = P.set_index(["unit", "contingency", "rank"]).performance
rows = []
pa = TRI.groupby(["pair", "contingency", "rank", "genotype"]).duration.mean().reset_index()
pb, pc, ra, rc = [], [], [], []
for s in S:
    key = ("-".join(s["pair"]), s["contingency"], s["rank"])
    mt = np.sort(s["match_t"])
    pb.append(dict(pair=key[0], contingency=key[1], rank=key[2], genotype=s["genotype"], duration=np.diff(mt).mean() if len(mt) > 1 else np.nan))
    allv = np.sort(np.concatenate([r["vs"][1:] for r in s["rats"].values()]))
    pc.append(dict(pair=key[0], contingency=key[1], rank=key[2], genotype=s["genotype"], duration=np.diff(allv).mean() if len(allv) > 1 else np.nan))
    for r in s["rats"].values():
        rc.append(dict(pair=key[0], contingency=key[1], rank=key[2], genotype=s["genotype"], duration=np.diff(r["vs"]).mean() if len(r["vs"]) > 1 else np.nan))
ra = TRI.groupby(["follower", "pair", "contingency", "rank", "genotype"]).duration.mean().reset_index()
for reading, pts, d in (("a", "pair", pa), ("a", "rat", ra), ("b", "pair", pd.DataFrame(pb)), ("c", "pair", pd.DataFrame(pc)), ("c", "rat", pd.DataFrame(rc))):
    d = d.copy()
    d["performance"] = [perf.get((p, c, k), np.nan) for p, c, k in zip(d.pair, d.contingency, d["rank"])]
    d = d.dropna(subset=["duration", "performance"])
    for g in ("WT", "Fmr1"):
        dd = d[d.genotype == g]
        r, p = stats.pearsonr(dd.duration, dd.performance)
        rows.append(dict(trial=reading, points=pts, genotype=g, n=len(dd), R=r, p=p, in_tol=abs(r - (-0.51 if g == "WT" else -0.40)) <= (0.051 if g == "WT" else 0.04)))
t7c = pd.DataFrame(rows)
print(t7c.round(4).to_string())

**Result: partially reproduced.** Session mean trial duration and performance are negatively correlated in both genotypes under every reading (R from -0.31 to -0.70, all p below 0.0001). No reading puts both R within tolerance: with asynchronous trials and one point per pair and session R is -0.39 (WT) and -0.51 (Fmr1), per rat and session -0.31 and -0.42 (only Fmr1 within tolerance).

## fig7e-gaze-final-quartile

> Follower-only gaze toward the leader increases from the first to the later quartiles of a trial and is
> greatest just before departure; mutual gaze shows no quartile effect, and there is no genotype effect.

Mean per rat (as follower) and quartile over its asynchronous trials of all analysed sessions of both
contingencies; mixed ANOVA (genotype between, quartile within, rat as subject); Holm-corrected
paired t tests of quartile 1 against 2, 3 and 4.

In [ ]:
import pingouin as pg
G = []
for (rat, g), d in TRI.groupby(["follower", "genotype"]):
    for qq in range(1, 5):
        G.append(dict(rat=rat, genotype=g, quartile=qq, gF=d[f"gF_q{qq}"].mean(), gM=d[f"gM_q{qq}"].mean()))
G = pd.DataFrame(G)
print(G.groupby(["genotype", "quartile"])[["gF", "gM"]].mean().round(4).to_string())
for dv in ("gF", "gM"):
    a = pg.mixed_anova(data=G, dv=dv, within="quartile", between="genotype", subject="rat", effsize="np2")
    print(dv); print(a.round(5).to_string())
ph = pg.pairwise_tests(data=G, dv="gF", within="quartile", subject="rat", padjust="holm")
print(ph[["A", "B", "T", "p_unc" if "p_unc" in ph else "p-unc", "p_corr" if "p_corr" in ph else "p-corr"]].round(5).to_string())
print("quartile with the highest follower-only gaze, per genotype:", G.groupby(["genotype", "quartile"]).gF.mean().groupby("genotype").idxmax().to_dict())
# the same, restricted to the aligned pair sessions (head positions agree with the pokes, data check)
TRI["aligned"] = [(p, c, k) in ALIGNED for p, c, k in zip(TRI.pair, TRI.contingency, TRI["rank"])]
Gal = []
for (rat, g), d in TRI[TRI.aligned].groupby(["follower", "genotype"]):
    for qq in range(1, 5):
        Gal.append(dict(rat=rat, genotype=g, quartile=qq, gF=d[f"gF_q{qq}"].mean(), gM=d[f"gM_q{qq}"].mean()))
Gal = pd.DataFrame(Gal)
print(f"aligned sessions only: {int(TRI.aligned.sum())} trials, rats per genotype {Gal.groupby('genotype').rat.nunique().to_dict()}")
print(Gal.groupby(["genotype", "quartile"])[["gF", "gM"]].mean().round(4).to_string())
for dv in ("gF", "gM"):
    a = pg.mixed_anova(data=Gal, dv=dv, within="quartile", between="genotype", subject="rat", effsize="np2")
    print(dv, "(aligned)"); print(a.round(5).to_string())
ph = pg.pairwise_tests(data=Gal, dv="gF", within="quartile", subject="rat", padjust="holm")
print(ph[["A", "B", "T", "p_unc" if "p_unc" in ph else "p-unc", "p_corr" if "p_corr" in ph else "p-corr"]].round(5).to_string())
print("quartile with the highest follower-only gaze, per genotype (aligned):", Gal.groupby(["genotype", "quartile"]).gF.mean().groupby("genotype").idxmax().to_dict())

**Result: reproduced (resting on an assumption).** Follower-only gaze rises over the trial in both genotypes (Fmr1 0.085 to 0.139, WT 0.119 to 0.156), the quartile effect is F(3,54) = 19.3, p below 0.001 (Greenhouse-Geisser p = 0.00006, partial eta squared 0.52 against the paper's 0.40), quartile 4 is highest in both genotypes, and the Holm-corrected tests put quartile 1 below quartiles 2, 3 and 4 (p = 0.00007, 0.0006, 0.0006). Mutual gaze has no quartile effect (F(3,54) = 1.26, p = 0.30). Genotype and its interactions are not significant (p = 0.22, 0.58; mutual 0.78, 0.95). The trials rest on this attempt's reading of the asynchronous trial. The head positions agree poorly with the pokes in 6 of 10 pairs (data check). In the aligned sessions only (2 pairs per genotype, 8 rats, 4,964 trials) the follower-only quartile effect holds (F(3,18) = 19.4, p = 0.00001; Holm p = 0.005, 0.001 and 0.005 for quartile 1 against 2, 3 and 4), but quartile 3 is highest in WT rats (0.147 against 0.140 in quartile 4) and mutual gaze has a quartile effect (F(3,18) = 10.1, p = 0.0004, Greenhouse-Geisser p = 0.007), so two parts of the criterion fail there. The criterion's reading is all analysed sessions, and the outcome is kept on it; it rests on the positions of the six pairs whose positions disagree with the pokes.

## fig7g-gaze-100-late and fig7h-gaze-50-fmr1-follower

Binomial GLMM of match on each asynchronous trial, logit(P(match)) ~ (gF + gM) x genotype x block,
random intercepts for the follower rat and the session (Eq. 12), fitted by variational Bayes
(`BinomialBayesMixedGLM`; glmmTMB is not in the environment). The model is written so that each
Fmr1 minus WT gaze slope in a block is one coefficient (the between-genotype contrast of emtrends in
that block; with one contrast per block the Tukey adjustment changes nothing). Readings: blocks are
thirds of the sessions (100%: 7, 7, 7; 50%: 12, 12, 11 or 11, 12, 12; every pair has the same
session count, so thirds by the pair's own count are the same); gF and gM as fractions of the trial,
or z-scored.

In [ ]:
def gaze_glmm(c, cuts, scale, data=None):
    data = TRI if data is None else data
    d = data[data.contingency == c].dropna(subset=["gF", "gM"]).copy()
    d["block"] = np.select([d["rank"] <= cuts[0], d["rank"] <= cuts[1]], [1, 2], 3)
    d["g"] = (d.genotype == "Fmr1").astype(int)
    for v in ("gF", "gM"):
        d[v + "x"] = d[v] if scale == "fraction" else (d[v] - d[v].mean()) / d[v].std()
    f = "match ~ C(block) * g + gFx:C(block) + gFx:C(block):g + gMx:C(block) + gMx:C(block):g"
    m = BinomialBayesMixedGLM.from_formula(f, {"rat": "0 + C(follower)", "session": "0 + C(sess)"}, d)
    fit = m.fit_vb()
    out = []
    for v in ("gF", "gM"):
        for b in (1, 2, 3):
            name = f"{v}x:C(block)[{b}]:g"
            k = m.exog_names.index(name)
            z = fit.fe_mean[k] / fit.fe_sd[k]
            out.append(dict(contingency=c, blocks="-".join(map(str, cuts)), scale=scale, gaze=v, block=["early", "mid", "late"][b - 1],
                            contrast=fit.fe_mean[k], se=fit.fe_sd[k], p=2 * stats.norm.sf(abs(z)), n=len(d)))
    return out

In [ ]:
t7g = pd.DataFrame([r for scale in ("fraction", "z") for r in gaze_glmm("rc100", (7, 14), scale)])
print(t7g.round(4).to_string())

**Result: not reproduced (resting on an assumption).** At 100% the late Fmr1 minus WT contrast for follower-only gaze is negative (-0.57 as a fraction, p = 0.11; -0.13 z-scored, p = 0.14), and for mutual gaze it is small and not significant (0.19, p = 0.73; 0.03, p = 0.72). The early and mid follower-only contrasts are not significant (p 0.07 to 0.77). In the aligned sessions only (cell below) the late contrasts are not significant either (follower-only -0.92, p = 0.097; mutual 1.79, p = 0.29).

In [ ]:
t7h = pd.DataFrame([r for cuts in ((12, 24), (11, 23)) for scale in ("fraction", "z") for r in gaze_glmm("rc50", cuts, scale)])
print(t7h.round(5).to_string())

In [ ]:
# fig7g and fig7h restricted to the aligned pair sessions
TRIal = TRI[TRI.aligned]
t7al = pd.DataFrame([r for scale in ("fraction",) for r in gaze_glmm("rc100", (7, 14), scale, TRIal)] +
                    [r for cuts in ((12, 24), (11, 23)) for r in gaze_glmm("rc50", cuts, "fraction", TRIal)])
print("aligned sessions only, fractions:")
print(t7al.round(5).to_string())

**Result: not reproduced (resting on an assumption).** At 50% only the early follower-only contrast is positive and significant (0.48 to 0.49 as a fraction, p = 0.016 and 0.017; 0.11 z-scored, p = 0.029 and 0.030). The mid and late contrasts are negative under every block and scaling reading (-0.06 to -0.43). The mutual-gaze contrast is significant early under the 11-12-12 blocks (p = 0.003). In the aligned sessions only, no follower-only contrast is significant (p 0.17 to 0.96), and the late mutual-gaze contrast is (2.79, p = 0.006, 11-12-12 blocks).

## Blocked claims

- figs1-mixed-pairs-impaired, fig2c-opaque-barrier-mixed-pairs: the dandiset holds no mixed WT-Fmr1 pair.
  Its 20 subjects form 10 pairs, each of one genotype (data check above).
- fig2b-opaque-barrier-lowers-performance, fig2c-opaque-barrier-mixed-pairs, and the sensory-deprived part
  of figs2-jaccard-similarity: no session is marked as sensory deprived. Every file's description names
  only the 100% or the 50% contingency, and the files carry no barrier or noise field. The paper says
  the sensory-deprived sessions were interspersed among 50% sessions; the cell below reads every file not
  analysed above (50% sessions after the 35th, 100% sessions after the 21st) to see whether they could
  be there unlabelled.
- fig2e-visual-cue-learning-similar, fig2f-cue-latency-wt-shorter: the visual-cue association task (single
  rats, LED cues) is not in the dandiset.
- fig3g-virtual-agent-performance-similar, fig3h-virtual-agent-efficiency-similar: the virtual-agent task
  (single rats) is not in the dandiset.
- fig6-marl-partner-aware: a simulation that uses no recorded data, with no published code; the learning
  rate, discount factor, timesteps per episode, number of runs, state encoding and the definition of
  "coordination efficiency (% of optimal matches)" are not given, and the reported 70% depends on them.
  Not run.
- figs3-leadership-history: blocked by its criterion until the supplement is read; the readings it names
  are shown above, not judged.

In [ ]:
# The sessions not analysed above: every 50% session after the 35th, and every 100% session after the 21st,
# read without positions. Could sensory-deprived sessions be among them, unlabelled?
allrows = coop.analysed_rows(coop.session_table(assets), extra=1000)
later = [r for r in allrows if r["rank"] > coop.N_SESSIONS[r["contingency"]]]
lfiles = coop.read_many([r["asset_id"] for r in later], positions=())
print(f"files not analysed above: {len(later)}; read: {len(lfiles)}")
print("descriptions:", collections.Counter(re.sub(r"subject \S+ paired with \S+", "subject X paired with Y", f["description"]) for f in lfiles))
print("task table fields:", collections.Counter(tuple(sorted(f["task_fields"].items())) for f in lfiles + files))
print("behavioral event series in these files:", sorted(set(k for f in lfiles for k in f["events"])))
LS = coop.build(later, lfiles)
first50 = {}
for s in S + LS:
    if s["contingency"] == "rc50":
        first50[s["pair"]] = min(first50.get(s["pair"], "9999"), s["date"])
SD = []
for s in LS:
    tr = sum(max(len(r["vw"]) - 1, 0) for r in s["rats"].values())
    SD.append(dict(pair="-".join(s["pair"]), genotype=s["genotype"], contingency=s["contingency"], rank=s["rank"], session=s["session"], date=s["date"],
                   after_first_50=s["date"] >= first50[s["pair"]], minutes=round((s["t1"] - s["t0"]) / 60, 1), matches=len(s["match_t"]),
                   rewarded=len(s["rewarded_t"]), transitions=tr, performance=round(200 * len(s["match_t"]) / max(tr, 1), 1)))
SD = pd.DataFrame(SD)
print("first 50% session date per pair:", {"-".join(k): v for k, v in sorted(first50.items())})
print(SD.groupby(["genotype", "pair", "contingency"]).agg(sessions=("rank", "size"), from_rank=("rank", "min"), to_rank=("rank", "max"), first=("date", "min"), last=("date", "max"),
                                                       in_50_period=("after_first_50", "sum"), performance_median=("performance", "median"),
                                                       performance_min=("performance", "min")).to_string())
print("performance of the last 9 analysed 50% sessions per pair (median, min):",
      P[(P.contingency == "rc50") & (P["rank"] > 26)].groupby("unit").performance.agg(["median", "min"]).round(1).to_dict("index"))
print("every session not analysed above, in date order:")
print(SD.sort_values(["pair", "date", "contingency", "rank"]).to_string(index=False))

**Result: blocked (metadata insufficient).** The 646 files not analysed above (416 at 50%, 230 at 100%) carry the same descriptions and task fields as the analysed ones, with no barrier, noise or sensory-deprived label, and the same event series. They include up to 53 further 50% sessions per pair (pairs FXM102-FXM103, FXM105-FXM107, FXM108-FXM109, XFN1-XFN3, XFN2-XFN4, ER1-ER2, FXM101-FXM110, FXM104-FXM106) and 100% sessions dated after the pair's first 50% session (17 for ER1-ER2, 2 to 5 for FXM102-FXM103, FXM105-FXM107 and FXM108-FXM109). Some later 50% sessions have much lower performance than the pair's last analysed sessions (minimum 0.0 for FXM105-FXM107, 15.4 for FXM102-FXM103, 14.6 for XFN2-XFN4), which sensory-deprived sessions could explain but nothing in the files confirms. Whether the sessions are there unlabelled cannot be decided from the files, so the reason is missing metadata. fig2c-opaque-barrier-mixed-pairs stays blocked as data missing, as it needs the mixed pairs. Some low sessions are also inside the analysed 50% sessions: the last 9 analysed 50% sessions of FXM105-FXM107 include one at 10.0% (median 39.4%). If sensory-deprived sessions were numbered as 50% sessions, some could be among the first 35 of a pair and would enter every 50% result; the files cannot settle this, and it is a question for the authors (findings).

In [ ]:
print(f"notebook run time: {(time.time() - T_START) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| fig1e-match-rate-wt-higher | blocked, methods underspecified (ANOVA readings disagree) |
| fig1f-transition-rate-wt-higher | blocked, methods underspecified (ANOVA readings disagree) |
| fig1g-match-transition-regression | partially reproduced (assumption) |
| fig1h-above-chance-late | partially reproduced (assumption) |
| fig1i-performance-wt-higher | blocked, methods underspecified (ANOVA readings disagree) |
| fig1j-partner-location-predicts-choice | not reproduced (assumption) |
| figs1-mixed-pairs-impaired | blocked, data missing |
| fig2b-opaque-barrier-lowers-performance | blocked, metadata insufficient |
| fig2c-opaque-barrier-mixed-pairs | blocked, data missing |
| fig2e-visual-cue-learning-similar | blocked, data missing |
| fig2f-cue-latency-wt-shorter | blocked, data missing |
| fig3c-triplets-by-contingency | not reproduced (assumption) |
| fig3d-entropy-wt-lower | partially reproduced (assumption) |
| fig3e-exploration-efficiency-wt-higher | not reproduced (assumption) |
| fig3g-virtual-agent-performance-similar | blocked, data missing |
| fig3h-virtual-agent-efficiency-similar | blocked, data missing |
| figs2-jaccard-similarity | not reproduced (assumption) |
| fig4b-arrival-lag-wt-shorter | not reproduced |
| fig4c-departure-lag-wt-shorter | not reproduced |
| fig4d-dwell-time | reproduced (assumption) |
| fig4eg-cross-correlation-wt-higher | not reproduced |
| fig4h-peak-lag-distribution | not reproduced (assumption) |
| fig4i-peak-correlation-wt-higher | blocked, methods underspecified (ANOVA readings disagree) |
| fig5d-leading-near-half | reproduced |
| fig5e-leading-difference-50 | not reproduced |
| figs3-leadership-history | blocked, methods underspecified (by the claims file) |
| fig6-marl-partner-aware | blocked, methods underspecified |
| fig7b-fmr1-more-asynchronous-matches | not reproduced (assumption) |
| fig7c-trial-duration-vs-performance | partially reproduced |
| fig7e-gaze-final-quartile | reproduced (assumption) |
| fig7g-gaze-100-late | not reproduced (assumption) |
| fig7h-gaze-50-fmr1-follower | not reproduced (assumption) |
| fig8b-partner-history-cooperation | blocked, other (the test is decided by how trials are built) |
| fig8d-fmr1-longer-delta-t | blocked, other (depends on fig8b) |
| fig8e-glmm-partner-cooperation | blocked, other (depends on fig8b) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The paper's analysis code is not published, so every event definition is this attempt's reading of the Methods: a visit is a run of pokes at one well; synchronous and asynchronous matches, asynchronous gaze trials and follower trials are defined from the poke sequences as stated in their sections.
- Pokes within 0.5 s of a poke at another well by the same rat are dropped as sensor crosstalk (an assumption; 4.8% of pokes, 140 of 1,120 rat sessions changed). The raw sequences are shown for transitions and performance.
- The ANOVA the legends call "Two-way RM ANOVA" has df that fit an ordinary two-way ANOVA with sessions as replicates (reading A). With pairs or rats as subjects (reading B) most genotype effects are not significant. The claims file blocks a claim where the two disagree, which decides four claims.
- The permutation test of Figure 1H uses matches rebuilt from the pokes (6 s hold, no match at the previous match's well), because the dataset's matches cannot be recomputed under a shift.
- The 1,000-shuffle null of Figure 1J uses the class a one-hot logistic regression predicts (the modal next well per partner well), which agrees with scikit-learn on 98% of real folds; the real accuracies are scikit-learn's.
- The GLMMs are fitted by variational Bayes in statsmodels (no R, glmmTMB or pymer4 in the environment); for Figure 8E a maximum-likelihood logistic regression with standard errors clustered by rat is shown beside it.
- Follower trials (Figure 8) are built from the arrivals as the Methods describe them: the first rat to arrive at a new well leads, and the other rat's next arrival is its choice. On sessions with one rat shifted in time this rule, like the first round's, gives a partner-history effect without any coupling, so Figures 8B, 8D and 8E are blocked.
- The 40 px radius of the position-based measures is taken in each video's own pixels (6.4 cm for most rats, 10 cm for FXM105, FXM107, FXM108 and FXM109); arrival and departure lags by head position are shown with and without the four pair sessions that have a lag over 100 s. The outcomes of Figures 4B and 4C use every session, as the claims file measures the lag, and both readings fail; with the removal, chosen after seeing one session with a mean lag of 803 s, the readings would disagree and the claims would be blocked, as they were before the third verification round.
- Pokes are also preferred over head position for dwell time (Figure 4D). Results restricted to the aligned pair sessions are shown for Figures 1J, 4, 7E, 7G and 7H; in them the Figure 7E pattern holds for follower-only gaze, but mutual gaze also rises over the quartiles.

What was found about the dataset:

- It holds the 10 same-genotype pairs (5 WT, 5 Fmr1) under 100% and 50% reward. The mixed pairs, the visual-cue task and the virtual-agent task are not in it. No session is labelled sensory deprived; if such sessions are among the 646 later files, nothing marks them.
- The camera device gives 0.0016 m per pixel in every file, but the processed positions of FXM105, FXM107, FXM108 and FXM109 were converted at 0.0025 m per pixel (regression of the processed on the raw DeepLabCut head position); FXM102 and FXM103 files hold no raw pose.
- In 6 of 10 pairs the head position agrees poorly with the pokes: at the poke times only 39 to 62% of head positions (subject means) are within 20 cm of the poked well's centre, against 99.9% or more in FXM101-FXM110, FXM104-FXM106, FXM201-FXM202 and FXM205-FXM206; no time shift of up to 30 s improves it. Position-based results (Figures 4B to 4I, 7) rest on these positions; results restricted to the 189 aligned pair sessions are shown for Figure 4.
- About 4.8% of pokes are near-simultaneous pokes at two wells by one rat (up to 128,979 pokes and 2,447 transitions in one rat session), which inflate transition counts unless removed.
- The matched pokes never repeat the well of the previous match (0 of 19,602), a rule the paper does not state for the rat task; poke offsets are not recorded.
- Session numbers have gaps for three pairs, and sessions run from 10.7 to 57.2 min against the paper's 20 to 30 min; 6.6% of rat sessions have pokes outside the session epoch.
- Both files of a pair session hold the same events; each recording was counted once. In every 7th pair session (80 of 560) positions and pokes are consistent with one clock (head 2.7 cm from the poked well at the poke, 30 to 46 cm when shifted by 5 to 10 s), and over all rat sessions the shift of the poke times (-30 to +30 s) that best fits is 0 s in the median for 9 of 10 pairs (-5 s for XFN2-XFN4), with the poor agreement in six pairs noted above.